# Hawkes $k$-Point Function — **Quadratic** $\varphi(v)=a v^2$, exp filter $\tau_g$ (tree + 1-loop test)

Streamlined notebook: model → propagator → diagrams → Phase J time-domain integration → simulation comparison.
No frequency-domain Phase I code.


In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from IPython.display import display, Math
from sage.all import (
    SR, matrix, latex, I, pi, exp, diff, solve, integrate, oo,
    dirac_delta, function, var
)

from msrjd.core.field_theory import FieldTheory, fourier_transform, inverse_fourier_transform
from msrjd.core.vertices import extract_vertex_types, extract_source_types, available_degrees
from msrjd.core.cache import PipelineCache
from msrjd.enumeration.loop_diagram_enumeration import enumerate_all
from msrjd.diagrams.filter import filter_prediagrams, classify_prediagram_vertices
from msrjd.diagrams.type_assignment import (
    enumerate_typed_diagrams, enumerate_all as enumerate_all_typed,
    build_field_index_map, TypedDiagram
)
from msrjd.diagrams.causality import filter_causal
from msrjd.diagrams.symmetry import (
    combinatorial_factor, compute_all_combinatorial_factors,
    deduplicate_typed_diagrams,
)

from models.hawkes_quad_expg import HAWKES_MODEL

print('Imports loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Configuration — all user-facing knobs live here
# ══════════════════════════════════════════════════════════════

# ── Theory ─────────────────────────────────────────────────────
k       = 4      # k-point cumulant (number of external legs)
max_ell = 1      # maximum loop order (0 = tree only, 1 = tree + 1-loop, ...)

# External fields: one per external leg (must have length k).
# Pick from the physical fields of the model: ('dn', pop), ('dv', pop)
# where pop is a population index (1, 2, ...).
# Examples:
#   k=1:  [('dn', 1)]
#   k=2:  [('dn', 1), ('dn', 2)]
#   k=2:  [('dn', 1), ('dv', 1)]    (mixed correlator)
external_fields = [('dn', 1), ('dn', 2), ('dv', 1), ('dv', 2)]

USE_CACHE = False  # True: pull from cache when available; False: recompute all

# Propagator cache for cell 8 (independent of USE_CACHE because it
# is keyed ONLY on HAWKES_MODEL + taylor_order, not on k /
# external_fields / fundamental parameters).  On first run, cell 8
# saves the symbolic K_ft, G_ft, adj_ft, D_delta matrices to
# saved_theories/<model-tag>_taylor<k>/propagator.sobj.  Subsequent
# kernel restarts load them in a few seconds instead of recomputing
# the expensive adjugate / inverse / D_delta-limit loop (~8 min for
# the quadratic Hawkes + exp filter).  Flip to False to force a
# fresh recompute (e.g., after editing the model file in place).
PROPAGATOR_CACHE_ENABLED = True

# ── Physical parameters (fundamental) ──────────────────────────
# Used by cell 7.4 (mean-field solve) and downstream numerical eval.
# Cell 23 reads this dict; if you prefer per-cell defaults, leave
# fundamental = None and cell 23 will fall back to its own block.
fundamental = {
    'E':     [1.2, 1.4],                        # external drive per population
    'w':     [[0.5, -0.1], [0.5, 0.4]],         # synaptic weights w_ij
    'tau':   10.0,                              # membrane time constant
    'a':     0.1,                               # gain of phi(v) = a * v^2
                                                # (quadratic -- pick small
                                                #  so n* stays O(1))
    'tau_g': 5.0,                               # exp synaptic filter timescale
}

# ── Simulation mode (only runs for k ≤ 4; theory runs for any k) ──
# 'off'   → skip all simulation cells; theory-only run.
# 'point' → evaluate theory + sim at TEST_POINTS below (fast).
# 'slice' → evaluate theory + sim on a τ grid (slower, produces plots).
SIM_MODE = 'point'

# Test points for 'point' mode. Each entry is a (k-1)-tuple of τ values
# (one per non-origin external leg).  Used only when SIM_MODE == 'point'.
# When SIM_MODE != 'point' this is ignored.
TEST_POINTS = [(4.0, 2.0, 2.0)]

# Slice-mode configuration (used only when SIM_MODE == 'slice').
TAU_FIXED_LIST = [1.0, -1.0]  # fixed τ values per non-swept axis at k ≥ 3
tau_max  = 50.0               # slice covers τ ∈ [-tau_max, +tau_max]
tau_step = 0.5                # grid spacing

# ── Simulation run parameters ──────────────────────────────────
N_RUNS    = 5                    # independent sim runs (more → tighter SEM)
T_sim     = float(5_000_000)     # total time per run
dt_sim    = float(0.01)          # Euler timestep
dt_bin    = float(1.0)           # binning resolution (smaller → less bin bias)
# Seed is ALWAYS randomized per run — fixed seeds hide the true
# sim variance, which is especially important for k=4 where the
# 4-point moment estimator has large run-to-run variability.

# ── Parallel evaluation (Phase J, cell 8.1c) ────────────────────
# Cell 8.1c evaluates theory on a τ grid via ``total_C_batch``.  The
# batch evaluator can fan out across worker processes (fork Pool) or
# run single-threaded.  Toggle this to control compute consumption.
#
# True  → parallel; spawns up to TD_N_WORKERS fork workers.  Typical
#         7× wall-clock speedup on a 12-core machine (measured on
#         quadratic Hawkes k=3, 201 τ × 4 slices: 173 min → 24 min).
# False → serial single-process evaluation.  Slower but deterministic
#         in resource use — safe on shared machines or anytime you
#         want to know exactly how much compute the run consumes.
TD_PARALLEL = True

# Cap on worker-process count.  None → auto-pick
# ``min(os.cpu_count(), len(tau_grid))``.  Set an int to cap lower —
# e.g., 4 on a shared machine where you want to leave cores for other
# users.  Ignored when TD_PARALLEL = False.
TD_N_WORKERS = None

# ── Display toggles (default = current verbose behavior) ───────
# Flip any to False to suppress big outputs.
SHOW_MODEL_METADATA      = True   # cell 4: model dict summary
SHOW_PROPAGATOR_DETAILS  = True   # cell 8: symbolic K, Ĝ, poles, C_k, G^R
SHOW_PREDIAGRAM_PLOTS    = True   # cell 11, 15: prediagram graphs
SHOW_VERTEX_TYPE_LIST    = True   # cell 13: per-vertex-type details
SHOW_DIAGRAM_DETAILS     = True   # cell 20: per-diagram prefactor + edges
SHOW_PREFACTOR_LIST      = True   # cell 22: per-diagram prefactor summary


# ── Sanity checks / auto-guards ────────────────────────────────
assert len(external_fields) == k, (
    f'external_fields has {len(external_fields)} entries but k={k}')
assert SIM_MODE in ('off', 'point', 'slice'), (
    f"SIM_MODE must be 'off', 'point', or 'slice'; got {SIM_MODE!r}")

# Simulation machinery is validated for k ≤ 4 only.  For larger k the
# cumulant-subtraction formula grows (more partitions) and the sim
# estimator grows variance rapidly — the theory part still runs.
if k > 4 and SIM_MODE != 'off':
    print(f'NOTE: k={k} > 4, auto-setting SIM_MODE="off" '
          f'(sim is validated only for k ≤ 4).')
    SIM_MODE = 'off'

print(f'k = {k},  max loop order = {max_ell},  SIM_MODE = {SIM_MODE}')
print(f'External fields: {external_fields}')


---
## 1. Theory / Model Information

In [ ]:
m = HAWKES_MODEL
if SHOW_MODEL_METADATA:
    print(f"Model:  {m['name']}")
    print(f"Convention: {m['background_rate_convention']}")
    print()

    print('── Index sets ──')
    for name, vals in m['index_sets'].items():
        print(f'  {name}: {list(vals)}')
    print()

    print('── Response fields (not expanded — full integration variables) ──')
    for f in m['response_fields']:
        print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
    print()

    print('── Physical fields (expanded around MF background) ──')
    for f in m['physical_fields']:
        print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
    print()

    print('── Parameters ──')
    for p in m.get('parameters', []):
        idx = '(indexed)' if p.get('indexed') else '(scalar)'
        dom = f", domain={p['domain']}" if p.get('domain') else ''
        print(f"  {p['name']}  {idx}{dom}  — {p.get('description', '')}")
    print()

    print('── Kernels ──')
    for kern in m.get('kernels', []):
        print(f"  {kern['name']}  — {kern.get('description', '')}")
    print()

    print('── Operators ──')
    for o in m.get('operators', []):
        print(f"  {o['name']}  (latex: ${o.get('latex_name', o['name'])}$)  — {o.get('description', '')}")
    print()

    print('── Nonlinear functions ──')
    for f in m.get('functions', []):
        print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
    print()

    print('── Specializations ──')
    print('  φ quadratic (cubic and higher coefficients = 0)')
    print('  g = δ(t)  (instantaneous synaptic coupling)')

### 1.1 Expand the action and show all sectors

In [3]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print(f'Taylor order: {ft.taylor_order}')
print(f'Polynomial ring: {R}')
print(f'Ring generators: {R.gens()}')
print(f'Response generators (n_tilde={ft._n_tilde}): {R.gens()[:ft._n_tilde]}')
print(f'Physical generators: {R.gens()[ft._n_tilde:]}')
print()

passed = ft.sanity_check()
print()
ft.summary()

Taylor order: 4
Polynomial ring: Multivariate Polynomial Ring in nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2 over Symbolic Ring
Ring generators: (nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2)
Response generators (n_tilde=4): (nt1, nt2, vt1, vt2)
Physical generators: (dn1, dn2, dv1, dv2)

=== Sanity checks ===
  [PASS]  (n_tilde=0, n_phys=0)  constant term
  [PASS]  (n_tilde=1, n_phys=0)  tadpole — must vanish at MF saddle
  [PASS]  (n_tilde=0, n_phys=1)  linear physical-only — must vanish at EOM

=== Action sectors ===
  (n_tilde=1, n_phys=1)  [free action]:


<IPython.core.display.Math object>

  (n_tilde=1, n_phys=2)  [vertex (order 3)]:


<IPython.core.display.Math object>

  (n_tilde=2, n_phys=0)  [noise kernel]:


...

<IPython.core.display.Math object>

  (n_tilde=4, n_phys=1)  [vertex (order 5)]:


<IPython.core.display.Math object>

  (n_tilde=4, n_phys=2)  [vertex (order 6)]:


<IPython.core.display.Math object>

### 1.2 Propagator

Extract the kernel matrix $\mathbf{K}$ from the free action, Fourier transform, and invert.

In [ ]:
import signal

# Helper: gate symbolic display on SHOW_PROPAGATOR_DETAILS toggle.
def _show_math(*args, **kwargs):
    if SHOW_PROPAGATOR_DETAILS:
        display(Math(*args, **kwargs))
def _show_print(*args, **kwargs):
    if SHOW_PROPAGATOR_DETAILS:
        print(*args, **kwargs)

# ── Propagator cache (keyed only on HAWKES_MODEL + taylor_order) ──
# This cell's symbolic propagator matrices (K_ker, K_ft, G_ft, adj_ft,
# D_omega, D_delta) depend ONLY on the model specification (fields,
# action, kernel image) and the Taylor order passed to FieldTheory.
# They do NOT depend on k / ell / external_fields / parameter values,
# so we cache them separately from the k-specific diagram-enumeration
# cache used in cell 11.
#
# Set ``PROPAGATOR_CACHE_ENABLED = False`` in cell 2 to disable, or
# call ``_prop_cache.clear()`` from a cell to nuke the cache.  If you
# edit the model file IN PLACE (without renaming) the cache key does
# NOT auto-invalidate -- clear manually.
import re as _re
from msrjd.core.cache import PipelineCache

try:
    _propagator_cache_enabled = bool(PROPAGATOR_CACHE_ENABLED)
except NameError:
    _propagator_cache_enabled = True
_prop_tag = _re.sub(r'[^A-Za-z0-9]+', '_',
                    HAWKES_MODEL['name']).strip('_').lower()
_prop_cache_dir = f'saved_theories/{_prop_tag}_taylor{ft.taylor_order}'
_prop_cache = PipelineCache(_prop_cache_dir)

_prop = None
if _propagator_cache_enabled and _prop_cache.exists('propagator'):
    try:
        _prop = _prop_cache.load('propagator')
        print(f'[cell 8] Propagator loaded from cache: '
              f'{_prop_cache_dir}/propagator.sobj')
    except Exception as _e:
        print(f'[cell 8] Propagator cache load failed ({_e!r}); '
              f'recomputing.')
        _prop = None

if _prop is None:
    S_free = ft.free_action()
    ring_gen_names = [str(g) for g in R.gens()]

    # Use ring variable ordering (must match build_field_index_map)
    resp_names = ring_gen_names[:ft._n_tilde]
    phys_names = ring_gen_names[ft._n_tilde:]

    pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
    pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

    nf = len(resp_names)
    K_data = [[SR(0)] * nf for _ in range(nf)]
    for exp_vec, coeff in S_free.dict().items():
        row = col = None
        for idx in range(len(ring_gen_names)):
            if exp_vec[idx] > 0:
                if idx in pos_to_row: row = pos_to_row[idx]
                if idx in pos_to_col: col = pos_to_col[idx]
        if row is not None and col is not None:
            K_data[row][col] += SR(coeff)

    K_mat = matrix(SR, K_data)

    # Convert to kernel form
    Dt       = ns.Dt
    delta_D  = ns.delta_D
    delta_Dp = ns.delta_Dp

    def _to_kernel(c):
        c = SR(c)
        # NOTE: ns.g is no longer substituted in the time domain (its
        # frequency-space image is applied post-FT via the model's
        # 'kernel_ft_image' hook).  Treat it as a constant symbolic factor
        # through _to_kernel: if the entry has no Dt and no delta_D, wrap
        # it in delta_D so that FT gives the constant back (no spurious
        # 2*pi*delta(omega) from FT of a constant in time).
        if c.has(delta_D):
            return c
        p0   = c.subs({Dt: 0})
        rest = (c - p0).subs({Dt: delta_Dp})
        return p0 * delta_D + rest

    K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

    resp_sr  = [ns.nt[i] for i in ns.pop] + [ns.vt[i] for i in ns.pop]
    phys_sr  = [ns.dn[i] for i in ns.pop] + [ns.dv[i] for i in ns.pop]

    _show_print('Field ordering:')
    display(Math(r'\tilde{\mathbf{a}} = ' + latex(vector(resp_sr))
                 + r',\qquad \mathbf{a} = ' + latex(vector(phys_sr))))
    print()
    _show_print('Kernel matrix K (time domain):')
    # Cosmetic cleanup for display: drop the redundant delta_D factor from
    # terms whose coefficient carries an explicit kernel (e.g. ns.g).  The
    # _to_kernel wrapper multiplies every plain constant coefficient by
    # delta_D so that Sage's fourier_transform correctly returns it
    # (instead of 2*pi*delta(omega)).  When the coefficient is *already*
    # a time-domain kernel symbol like g(t), the delta_D is pure
    # bookkeeping and visually noisy, so we divide it out for display only.
    def _pretty_kernel_entry(e):
        e = SR(e).expand()
        kernels = [getattr(ns, ksp['name']) for ksp in HAWKES_MODEL.get('kernels', [])]
        if not kernels:
            return e
        op = e.operator()
        is_sum = op is not None and 'add' in getattr(op, '__name__', '').lower()
        terms = e.operands() if is_sum else [e]
        # Drop delta_D only if every nonzero term has both delta_D and a kernel.
        every_term_has_kernel_and_delta = all(
            (not SR(t).is_zero())
            and t.has(delta_D)
            and any(t.has(k) for k in kernels)
            for t in terms
        )
        if every_term_has_kernel_and_delta:
            try:
                return (e / delta_D).expand()
            except Exception:
                pass
        return e
    K_ker_display = K_ker.apply_map(_pretty_kernel_entry)
    display(Math(r'\mathbf{K} = ' + latex(K_ker_display)))

    # Fourier transform
    t_var = SR.var('t')
    omega = SR.var('omega', latex_name=r'\omega')

    time_domain = {
        delta_D:  dirac_delta(t_var),
        delta_Dp: diff(dirac_delta(t_var), t_var),
        # ns.g is NOT substituted here -- it is a constant symbolic factor
        # that passes through the Fourier transform, then is replaced by
        # its frequency-space image below.
    }

    K_ft_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            c = K_ker[i, j]
            if not c.is_zero():
                K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

    K_ft = matrix(SR, K_ft_data)

    # Apply the model's kernel frequency-space image: g -> g_hat(omega).
    # For g(t) = (1/tau_g) exp(-t/tau_g) Theta(t), the image is
    #   g_hat(omega) = 1 / (1 + i omega tau_g).
    # Any model that declares a 'kernel_ft_image' hook substitutes its
    # kernel symbols here; models that specialize g -> delta_D in their
    # time-domain specializations (e.g., the plain-delta linear Hawkes)
    # never reach this branch because ns.g is already gone.
    _kft_hook = HAWKES_MODEL.get('kernel_ft_image')
    if _kft_hook is not None:
        _kft_subs = _kft_hook(ns, omega)
        K_ft = K_ft.apply_map(lambda e: SR(e).subs(_kft_subs))
    _show_print('Fourier-domain kernel:')
    display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

    # Propagator inverse
    G_ft = K_ft.inverse().apply_map(lambda e: e.factor())
    G_ft_explicit = True
    _show_print('Propagator:')
    display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))

    # Adjugate and determinant (symbolic; the determinant is a RATIONAL
    # function in omega when the kernel image introduces denominators such
    # as (1 + i omega tau_g) for the exponential synaptic filter).
    adj_ft  = K_ft.adjugate()
    D_omega = K_ft.det()

    # Delta-function coefficients via polynomial division:
    #   G_ft[i,j](ω) = Q(iω) + R(ω)/D(ω)
    # where Q is the polynomial (non-proper) part and R/D is strictly proper.
    # Q maps to distributional terms: constant → δ(t), iω → δ'(t), etc.
    # The simplest case: D[i,j] = lim_{ω→∞} G_ft[i,j](ω) = coefficient of δ(t).
    # Use Sage's symbolic limit — works with symbolic parameters, no numerics needed.
    from sage.all import limit as _limit, oo as _oo
    D_delta_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            entry = SR(G_ft[i, j])
            if entry.is_zero():
                continue
            try:
                lim_val = _limit(entry, **{str(omega): _oo})
                if not lim_val.is_zero():
                    D_delta_data[i][j] = lim_val
            except Exception:
                pass
    D_delta = matrix(SR, D_delta_data)

    _show_print('Delta-function coefficient matrix:')
    display(Math(r'\mathbf{D} = ' + latex(D_delta)))

    # Save propagator to cache
    _prop = {
        'K_ker': K_ker, 'K_ft': K_ft, 'G_ft': G_ft,
        'adj_ft': adj_ft, 'D_omega': D_omega, 'D_delta': D_delta,
        't_var': t_var, 'omega': omega, 'nf': nf,
        'ring_gen_names': ring_gen_names,
    }
    if _propagator_cache_enabled:
        _prop_cache.save('propagator', _prop)
        print(f'[cell 8] Propagator cached to: '
              f'{_prop_cache_dir}/propagator.sobj')
else:
    # Cache hit: unpack into the names the rest of the pipeline
    # expects.  (Displays that happened on the original cache-miss
    # run are NOT re-run here -- flip PROPAGATOR_CACHE_ENABLED or
    # clear the cache if you want to inspect them again.)
    K_ker          = _prop['K_ker']
    K_ft           = _prop['K_ft']
    G_ft           = _prop['G_ft']
    adj_ft         = _prop['adj_ft']
    D_omega        = _prop['D_omega']
    D_delta        = _prop['D_delta']
    t_var          = _prop['t_var']
    omega          = _prop['omega']
    nf             = _prop['nf']
    ring_gen_names = _prop.get('ring_gen_names', [])

# Pole finding is DEFERRED to after num_params is known (cell 23).
# Symbolic solve() on the rational determinant -- or even on its
# cleared-denominator numerator polynomial (degree >= 4 in omega once
# the exp filter is in play) -- produces no closed-form quartic/quintic
# roots.  Pole-residue data is therefore populated numerically below via
# compute_poles_and_residues(num_params), called from cell 23.
pole_vals = None   # list of complex numbers, filled after num_params
C_mats    = None   # list of matrix(CDF, ...), filled after num_params
G_t       = None   # symbolic G_t requires symbolic poles; not built here

def compute_poles_and_residues(num_params_dict):
    """Substitute num_params into K_ft, find retarded poles and residue
    matrices.

    Characteristic-polynomial strategy: the poles of G(omega) = K_ft^{-1}
    are the zeros of det(K_ft).  However, Sage's det().numerator() on a
    rational determinant can carry extra denominator-cancelling factors,
    introducing spurious roots.  We instead read the characteristic
    polynomial from the LCM of G_ft entry denominators.

    To avoid SR-level rational-function fragility (``PolynomialRing(CDF,
    'omega')(expr)`` fails with ``fraction must have unit denominator``
    if ``expr`` has any symbolic denominator left -- even a trivial
    one), we do all the polynomial algebra in ``CDF[omega]`` from the
    start: coerce each G_ft entry into that ring's fraction field and
    accumulate a polynomial LCM.

    Residue theorem:  G has a simple pole at omega_k with residue
        C_k = i * adj(omega_k) / det'(omega_k)
    where adj and det' are evaluated via SR substitution at omega_k.
    Results are returned as complex floats / CDF matrices.
    """
    from sage.all import CDF, PolynomialRing

    # Substitute numerical parameters into the symbolic matrices
    K_ft_num   = K_ft.apply_map(lambda e: SR(e).subs(num_params_dict))
    adj_ft_num = adj_ft.apply_map(lambda e: SR(e).subs(num_params_dict))
    G_ft_num   = G_ft.apply_map(lambda e: SR(e).subs(num_params_dict))

    # Build the characteristic polynomial in CDF[omega] directly.
    #
    # The char-poly of a non-degenerate rational-matrix G(omega) is the
    # LCM of its entry denominators, but numerical LCM in CDF[omega] is
    # fragile: floating-point noise defeats the GCD that LCM would use
    # to fold common factors, so iterating ``char_poly.lcm(den_p)``
    # across all 16 entries blows up into a product-of-all-denominators
    # polynomial with spurious additional roots.
    #
    # For a non-degenerate propagator at least one G_ft entry carries
    # the FULL characteristic polynomial in its denominator (the
    # diagonal entries of K^{-1} do, when no numerator factor cancels
    # a pole).  So we pick the highest-degree entry-denominator as the
    # char poly and trust that to carry the full pole structure.  If a
    # pole happened to cancel from ALL entries it also drops out of
    # the correlator -- not a physical pole.
    PR = PolynomialRing(CDF, 'omega')
    FR = PR.fraction_field()
    char_poly = PR(1)
    for i in range(nf):
        for j in range(nf):
            entry = SR(G_ft_num[i, j])
            if entry.is_zero():
                continue
            try:
                den_p = PR(entry.denominator())
            except Exception:
                try:
                    rat = FR(entry)
                    den_p = rat.denominator()
                except Exception:
                    continue
            if den_p.degree() > char_poly.degree():
                char_poly = den_p

    roots_all = [complex(r) for r in char_poly.roots(multiplicities=False)]

    # Retarded: keep Im(omega) > 0, deduplicate
    pruned = []
    for r in roots_all:
        if r.imag <= 1e-9:
            continue
        if any(abs(r - q) < 1e-7 for q in pruned):
            continue
        pruned.append(r)

    # Residue matrices: C_k = i * adj(omega_k) / D'(omega_k)
    D_rational = K_ft_num.det()
    D_prime_expr = diff(D_rational, omega)
    C_mats_local = []
    for omega_k in pruned:
        C_data = [[CDF(0)] * nf for _ in range(nf)]
        den = complex(D_prime_expr.subs({omega: omega_k}))
        for i in range(nf):
            for j in range(nf):
                n_ij = adj_ft_num[i, j]
                if not SR(n_ij).is_zero():
                    num = complex(n_ij.subs({omega: omega_k}))
                    C_data[i][j] = CDF(I) * CDF(num) / CDF(den)
        C_mats_local.append(matrix(CDF, C_data))
    return pruned, C_mats_local


print()
_show_print('Full retarded propagator:')
display(Math(
    r'\mathbf{G}^R(t) = \mathbf{D}\,\delta(t)'
    r' \;+\; \Theta(t) \sum_{k}'
    r' \mathbf{C}_k \, e^{i\omega_k t}'
))
print()
print('[cell 8] Pole finding deferred: will be computed numerically after '
      'num_params is defined (cell 23).')


---
## 2. Prediagram Enumeration

Enumerate all trees, topologies, and prediagrams for the $k$-point function
at each loop order $\ell = 0, \ldots, \ell_{\max}$.

In [5]:
from sage.plot.plot import graphics_array

def plot_prediagrams(pds, title_prefix=''):
    """Plot prediagrams with colored vertices."""
    if not pds:
        print('  (none)')
        return
    plots = []
    for i, (D, G, leaves, internal) in enumerate(pds):
        leaves_set = set(leaves)
        color_map = {}
        for v in D.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            elif D.in_degree(v) == 0:
                color_map.setdefault('red', []).append(v)
            else:
                color_map.setdefault('lightblue', []).append(v)
        plots.append(D.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

def plot_topologies(topos, title_prefix=''):
    """Plot undirected topologies."""
    if not topos:
        print('  (none)')
        return
    plots = []
    for i, (G, leaves, internal) in enumerate(topos):
        leaves_set = set(leaves)
        color_map = {}
        for v in G.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            else:
                color_map.setdefault('lightgray', []).append(v)
        plots.append(G.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

print('Plot helpers defined.')

Plot helpers defined.


In [ ]:
# ── Pipeline cache (depends on model + k) ──────────────────────
# Model name + external fields sanitized for filesystem use
_model_tag = HAWKES_MODEL['name'].replace(' ', '_').lower()
_ext_tag = '_'.join(f'{f[0]}{f[1]}' for f in external_fields)
cache_dir = f'saved_results/{_model_tag}_k{k}_{_ext_tag}'
cache = PipelineCache(cache_dir)
if USE_CACHE:
    print(f'Cache: {cache}')
    for e in cache.list_cached():
        print(f'  {e["key"]}  (saved {e["saved_at"][:19]})')
else:
    print('Cache disabled — all stages will be recomputed.')

# ── Enumerate prediagrams for each loop order ──────────────────
pds_by_ell = {}    # ell -> list of prediagrams
counts_by_ell = {} # ell -> counts dict

for ell in range(max_ell + 1):
    def _enumerate(ell=ell):
        t, tp, pd, c = enumerate_all(k, ell, verbose=False)
        return (pd, c)

    pds, counts = cache.get_or_compute(
        'prediagrams', _enumerate, k=k, loop_order=ell
    ) if USE_CACHE else _enumerate()

    pds_by_ell[ell] = pds
    counts_by_ell[ell] = counts

    print(f'ell={ell}:  {counts["n_trees"]} trees,  '
          f'{counts["n_topologies"]} topologies,  '
          f'{counts["n_prediagrams"]} prediagrams')
    if SHOW_PREDIAGRAM_PLOTS:
        plot_prediagrams(pds, title_prefix=f'ell={ell} PD ')

---
## 3. Vertex Extraction

Extract typed vertices (`VertexType`, `SourceType`) from the expanded action, based on the maximum vertex degree required by the prediagrams.

In [ ]:
from msrjd.enumeration.degree_scan import max_vertex_degree, scan_source_vertices

all_pds = [pd for pds in pds_by_ell.values() for pd in pds]
max_deg = max_vertex_degree(all_pds)
src_degs = scan_source_vertices(all_pds)
print(f'Max vertex degree across all prediagrams: {max_deg}')
print(f'Source vertex out-degrees needed: {src_degs}')
print(f'Current Taylor order: {ft.taylor_order}')
print(f'Sufficient: {ft.taylor_order >= max_deg}')
print()

vtypes = extract_vertex_types(ft)
stypes = extract_source_types(ft)

print(f'── Interaction vertex types: {len(vtypes)}')
print(f'── Source types: {len(stypes)}')
if SHOW_VERTEX_TYPE_LIST:
    print()
    for i, vt in enumerate(vtypes):
        print(f'  V{i+1}: bigrade={vt.bigrade}, in_deg={vt.in_degree}, out_deg={vt.out_degree}')
        print(f'        resp_legs={vt.response_legs}, phys_legs={vt.physical_legs}')
        display(Math(f'\\quad\\text{{coeff}} = {latex(vt.coefficient)}'))
    for i, st in enumerate(stypes):
        print(f'  S{i+1}: bigrade={st.bigrade}, out_deg={st.out_degree}')
        print(f'        resp_legs={st.response_legs}')
        display(Math(f'\\quad\\text{{coeff}} = {latex(st.coefficient)}'))

int_degs, src_degs_avail = available_degrees(vtypes, stypes)
print(f'\nAvailable interaction degrees (in, out): {sorted(int_degs)}')
print(f'Available source out-degrees: {sorted(src_degs_avail)}')

---
## 4. Prediagram Filtering

Remove prediagrams whose vertex degree signatures don't match any available vertex or source type.

In [ ]:
kept_by_ell = {}  # ell -> list of kept prediagrams

for ell in range(max_ell + 1):
    def _filter(ell=ell):
        kept, disc = filter_prediagrams(pds_by_ell[ell], vtypes, stypes)
        return (kept, disc)

    kept, disc = cache.get_or_compute(
        'filtered', _filter, k=k, loop_order=ell
    ) if USE_CACHE else _filter()

    kept_by_ell[ell] = kept
    print(f'ell={ell}:  {len(pds_by_ell[ell])} prediagrams -> '
          f'{len(kept)} kept, {disc} discarded')
    if SHOW_PREDIAGRAM_PLOTS:
        plot_prediagrams(kept, title_prefix=f'ell={ell} PD ')

---
## 5. Unique Typed Diagrams

For each surviving prediagram, enumerate all valid field-type assignments,
filter for causality, and deduplicate by isomorphism to obtain the set of
**unique Feynman diagrams** $\Gamma$.  Only these are cached — intermediate
typed diagrams are not stored.

In [ ]:
# External fields are specified in the Configuration cell above.
print(f'External fields (k={k}): {external_fields}')

# Build field index maps
ring_var_names_list = list(ns._ring_var_names)
n_tilde = ft._n_tilde
resp_idx, phys_idx = build_field_index_map(ring_var_names_list, n_tilde)

# Validate that each external field exists in the physical field index
for field in external_fields:
    assert field in phys_idx, (
        f'External field {field} not found in physical fields. '
        f'Available: {sorted(phys_idx.keys())}'
    )

print('\nResponse field index map:')
for leg, idx in sorted(resp_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> row {idx}')
print()
print('Physical field index map:')
for leg, idx in sorted(phys_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> col {idx}')

In [ ]:
unique_by_ell = {}  # ell -> list of unique TypedDiagram

for ell in range(max_ell + 1):
    def _unique(ell=ell):
        typed = enumerate_all_typed(
            kept_by_ell[ell], external_fields, vtypes, stypes,
            G_ft, resp_idx, phys_idx
        )
        causal, n_disc, _ = filter_causal(typed)
        unique = deduplicate_typed_diagrams(causal)
        print(f'  ell={ell}: {len(kept_by_ell[ell])} prediagrams -> {len(typed)} typed'
              f' -> {len(causal)} causal -> {len(unique)} unique')
        return unique

    unique_by_ell[ell] = cache.get_or_compute(
        'unique_typed', _unique, k=k, loop_order=ell
    ) if USE_CACHE else _unique()

    print(f'ell={ell}: {len(unique_by_ell[ell])} unique diagrams')

# Convenience aliases used downstream
unique_tree = unique_by_ell.get(0, [])
all_unique = [td for ell in range(max_ell + 1) for td in unique_by_ell.get(ell, [])]
print(f'\nTotal unique diagrams across all loop orders: {len(all_unique)}')

---
## 6. Diagram Details & Combinatorial Factor $M(\Gamma)$

For each unique diagram $\Gamma$, display the vertex assignments, edge types,
and the **combinatorial factor** $M(\Gamma)$ together with the scalar prefactor.

The vertex coefficients shown include the $(-1)$ from the partition function
convention $Z = \int \exp(-S)$.

In [ ]:
from msrjd.diagrams.symmetry import classify_coefficient_factors

# Get time-dependence metadata from the model
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure', {
    'temporal_type': 'white', 'amplitude_params': []
})
print(f'Noise temporal type: {noise_structure.get("temporal_type", "white")}')
print(f'Time-dependent params: {time_dep_params if time_dep_params else "(none -- stationary)"}')
print()


def show_unique_diagram(td, idx, winfo):
    """Display a unique diagram with M(Gamma) and weight structure."""
    M = winfo['M']
    print(f'\n{"="*60}')
    print(f'Unique Diagram {idx}    M = {M}')
    print(f'{"="*60}')

    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} <- {field[0]}{field[1]}')

    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            eff_coeff = -SR(vtype.coefficient)
            display(Math(f'\\quad (-1)\\cdot c_{{v_{v}}} = {latex(eff_coeff)}'))

    print('  Edges (propagator assignments):')
    for edge_key in sorted(td.edge_types.keys()):
        resp_leg, phys_leg = td.edge_types[edge_key]
        u, v = edge_key[0], edge_key[1]
        ri, pi = td.propagator_indices.get(edge_key, ('?', '?'))
        print(f'    {u} -> {v}:  {resp_leg[0]}{resp_leg[1]} -> '
              f'{phys_leg[0]}{phys_leg[1]}  [G_hat[{ri},{pi}]]')

    sp = winfo['scalar_prefactor']
    display(Math(
        r'\text{Scalar prefactor: }\;' + latex(sp)
    ))


# ── Display all unique diagrams by loop order ──
if SHOW_DIAGRAM_DETAILS:
    for ell in range(max_ell + 1):
        diagrams = unique_by_ell.get(ell, [])
        ell_label = 'TREE-LEVEL' if ell == 0 else f'{ell}-LOOP'
        print('='*60)
        print(f'{ell_label} UNIQUE DIAGRAMS ({len(diagrams)})')
        print('='*60)

        for i, td in enumerate(diagrams, 1):
            winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
            label = f'Tree-{i}' if ell == 0 else f'{ell}L-{i}'
            show_unique_diagram(td, label, winfo)

---
## 7. Propagator Data & Diagram Prefactors

Assemble the time-domain propagator data (poles, residues, $\delta$
coefficients from cell 1.2) and compute the scalar prefactor
$M(\Gamma) \times \prod_v (-c_v)$ for each unique typed diagram.

No frequency-domain integral construction is used.  The pipeline
goes directly from typed diagrams to time-domain vertex-time
integration.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.  Propagator data + diagram prefactors (pure time-domain path)
# ═══════════════════════════════════════════════════════════════
# This cell assembles everything needed for time-domain evaluation:
#   1. propagator_data: pole_vals, C_mats, D_delta (from cell 8)
#   2. Scalar prefactors for each typed diagram (from symmetry.py)
#   3. The diagram list itself (from cells 15-18)
#
# No frequency-domain integral construction is used.  The frequency-

from msrjd.diagrams.symmetry import classify_coefficient_factors

# ── Propagator data dict ──
# pole_vals / C_mats are None if cell 8 deferred them for numerical
# evaluation (needed whenever the kernel image introduces rational
# factors like 1/(1+i omega tau_g)).  Cell 23 fills them in via
# compute_poles_and_residues(num_params) once num_params is built.
propagator_data = {
    'pole_vals': pole_vals,
    'C_mats': C_mats,
    'D_delta': D_delta,
    'nf': nf,
}

if pole_vals is None:
    print('Propagator: poles deferred (filled numerically in cell 23)')
else:
    print(f'Propagator: {len(pole_vals)} poles, nf={nf}')
print(f'D_delta nonzero entries: {sum(1 for i in range(nf) for j in range(nf) if abs(complex(D_delta[i,j])) > 1e-15)}')

# ── Collect ALL unique typed diagrams across loop orders ──
all_unique = []
for ell in range(max_ell + 1):
    all_unique.extend(unique_by_ell.get(ell, []))
print(f'\nTotal unique diagrams: {len(all_unique)}')

# ── Compute scalar prefactors for each diagram ──
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure')

diagram_prefactors = []
for i, td in enumerate(all_unique):
    coeff_info = classify_coefficient_factors(
        td,
        time_dep_params=time_dep_params,
        noise_structure=noise_structure,
    )
    diagram_prefactors.append(coeff_info['scalar_prefactor'])

# ── Summary ──
from msrjd.integration.time_domain import _loop_number_from_graph
n_tree = sum(1 for td in all_unique if _loop_number_from_graph(td) == 0)
n_loop = len(all_unique) - n_tree
print(f'  Tree-level: {n_tree}')
print(f'  Loop-level: {n_loop}')

if SHOW_PREFACTOR_LIST:
    for i, (td, pf) in enumerate(zip(all_unique, diagram_prefactors)):
        ln = _loop_number_from_graph(td)
        D = td.prediagram[0]
        ext = {lf: td.external_legs[lf] for lf in td.prediagram[2]}
        label = f'Tree-{i+1}' if ln == 0 else f'{ln}L-{i+1}'
        print(f'  {label}: pf={pf}, ext={ext}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.4  Numerical evaluation — adaptive by k
# ═══════════════════════════════════════════════════════════════
import numpy as np
from scipy.optimize import fsolve
from sage.all import numerical_integral, CC, fast_callable, CDF, diff

# ═══════════════════════════════════════════════════════════════
# Fundamental parameters (read from cell 2; fallback defaults here)
# ═══════════════════════════════════════════════════════════════
# If `fundamental` was set in cell 2 (the config cell), use it.
# Otherwise fall back to the local defaults below.
try:
    fundamental  # set by cell 2
    if fundamental is None:
        raise NameError
except NameError:
    fundamental = {
        'E':   [1.0, 1.0],                        # external drive per population
        'w':   [[0.4, 0.5], [0.5, 0.4]],         # synaptic weights w_ij
        'tau': 10.0,                               # membrane time constant
    }

npop = len(ns.pop)

print('Fundamental parameters:')
for key, val in fundamental.items():
    print(f'  {key} = {val}')

# ═══════════════════════════════════════════════════════════════
# Solve mean-field equations from the model
# ═══════════════════════════════════════════════════════════════
# Read phi_concrete from model and differentiate symbolically
v_sym = SR.var('_v_mf_')
taylor_order = ft.taylor_order

# Build substitution dict: map model parameter symbols to fundamental values.
# This handles any model (linear phi has no 'a', quadratic has 'a', etc.)
_param_subs = {}
for pspec in HAWKES_MODEL.get('parameters', []):
    pname = pspec['name']
    if pname not in fundamental:
        continue
    val = fundamental[pname]
    if pspec.get('indexed', False):
        if isinstance(val, list) and val and isinstance(val[0], list):
            # 2D matrix-valued indexed param (e.g., synaptic weight w_{ij})
            # Substitute each w_{ij} symbol by name so this works even
            # if mf_substitutions has already replaced ns.w with a
            # nested SR matrix that is no longer hashable.
            for i in ns.pop:
                for j in ns.pop:
                    _param_subs[SR.var(f'{pname}{i+1}{j+1}')] = val[i][j]
        else:
            # 1D vector-valued indexed param (e.g., E_i, nstar_i)
            for i in ns.pop:
                _param_subs[SR.var(f'{pname}{i+1}')] = val[i]
    else:
        _param_subs[SR.var(pname)] = val

phi_derivs = {}  # phi_derivs[i][dk] = k-th derivative of phi_i(v), symbolic
for i in ns.pop:
    phi_expr = HAWKES_MODEL['phi_concrete'](ns, i, v_sym)
    phi_derivs[i] = {}
    for dk in range(taylor_order + 1):
        if dk == 0:
            phi_derivs[i][dk] = phi_expr
        else:
            phi_derivs[i][dk] = diff(phi_expr, v_sym, dk)

# Build numerical phi callable from the symbolic expression
def phi_num(i, v_val):
    """Evaluate phi_i(v) numerically."""
    return float(phi_derivs[i][0].subs(_param_subs).subs({v_sym: v_val}))

# Solve MF self-consistency: n*_i = phi_i(E_i + sum_j w_ij * n*_j)
def mf_residual(nstar_vec):
    residuals = []
    for i in ns.pop:
        v_star_i = fundamental['E'][i] + sum(
            fundamental['w'][i][j] * nstar_vec[j] for j in ns.pop
        )
        residuals.append(nstar_vec[i] - phi_num(i, v_star_i))
    return residuals

# Initial guess: small positive rates
nstar_guess = [1.0] * npop
nstar_sol = fsolve(mf_residual, nstar_guess, full_output=True)
nstar_vals = [float(x) for x in nstar_sol[0]]
info = nstar_sol[1]

# Compute v* and phi derivatives at the fixed point
vstar_vals = []
phi_deriv_vals = {}  # phi_deriv_vals[(dk, i)] = d^k phi_i / dv^k |_{v=v*}
for i in ns.pop:
    v_star_i = fundamental['E'][i] + sum(
        fundamental['w'][i][j] * nstar_vals[j] for j in ns.pop
    )
    vstar_vals.append(v_star_i)
    for dk in range(taylor_order + 1):
        phi_deriv_vals[(dk, i)] = float(
            phi_derivs[i][dk].subs(_param_subs).subs({v_sym: v_star_i})
        )

# Sanity check: phi_i(v*_i) should equal n*_i
print(f'\nMean-field solution:')
for i in ns.pop:
    phi_at_vstar = phi_deriv_vals[(0, i)]
    print(f'  pop {i+1}:  v* = {vstar_vals[i]:.6f},  '
          f'n* = {nstar_vals[i]:.6f},  '
          f'phi(v*) = {phi_at_vstar:.6f}  '
          f'{"OK" if abs(phi_at_vstar - nstar_vals[i]) < 1e-10 else "MISMATCH!"}')

print(f'\nDerived phi derivatives:')
for i in ns.pop:
    derivs_str = ', '.join(
        f"phi{dk}_{i+1} = {phi_deriv_vals[(dk, i)]:.6f}"
        for dk in range(1, taylor_order + 1)
        if (dk, i) in phi_deriv_vals
    )
    print(f'  pop {i+1}:  {derivs_str}')

# ═══════════════════════════════════════════════════════════════
# Assemble num_params for symbolic substitution
# ═══════════════════════════════════════════════════════════════
# Generic: sweep over every scalar parameter declared in the model,
# pulling its value from `fundamental` when present.  Indexed
# parameters (w, nstar, ...) expand into per-index SR symbols.
# Non-indexed parameters (tau, a, tau_g, ...) map to a single symbol.
num_params = {}

for pspec in HAWKES_MODEL.get('parameters', []):
    pname = pspec['name']
    if pname not in fundamental:
        # Parameter not supplied in `fundamental` -- skipped here and
        # filled from MF solve below (e.g., nstar, vstar).
        continue
    if pspec.get('indexed', False):
        val = fundamental[pname]
        if isinstance(val, list) and val and isinstance(val[0], list):
            # 2D matrix-valued param (e.g., w[i][j])
            for i in ns.pop:
                for j in ns.pop:
                    num_params[SR.var(f'{pname}{i+1}{j+1}')] = val[i][j]
        else:
            # 1D vector-valued param (e.g., E[i])
            for i in ns.pop:
                num_params[SR.var(f'{pname}{i+1}')] = val[i]
    else:
        # Scalar param (tau, a, tau_g, ...)
        num_params[SR.var(pname)] = fundamental[pname]

# Derived: nstar and vstar (from the MF solve above)
# -------------------------------------------------------------------
# Both nstar_i and vstar_i are MF-derived, not user-supplied in
# ``fundamental``, so the generic parameter loop above doesn't catch
# them.  They are needed to substitute into the kernel matrix K(omega)
# BEFORE pole finding -- e.g. the quadratic model's specialization
# ``phi1_i -> 2 a vstar_i`` runs after ``mf_bg_conditions``, leaving
# ``vstar_i`` as a free symbol in the numeric K_ft.  Without the
# substitution below, ``compute_poles_and_residues`` cannot coerce the
# characteristic polynomial to CDF[omega] and returns an empty pole
# list -- which in turn makes ``build_G_t_matrix`` return a zero
# ``smooth`` and crashes downstream ``G_t_entry`` calls.
for i in ns.pop:
    num_params[ns.nstar[i]] = float(nstar_vals[i])
    num_params[ns.vstar[i]] = float(vstar_vals[i])

# Derived: phi derivatives (phi1, phi2, ..., up to taylor_order)
for i in ns.pop:
    for dk in range(1, taylor_order + 1):
        sym = SR.var(f'phi{dk}_{i+1}')
        if (dk, i) in phi_deriv_vals:
            num_params[sym] = phi_deriv_vals[(dk, i)]

print(f'\nFull num_params:')
for sym, val in num_params.items():
    print(f'  {sym} = {val}')

# ── Tau grid for Phase J evaluation ──
tau_out = np.arange(-50.0, 50.25, 0.5)   # coarse grid for fast slice eval
tau_residue = tau_out.copy()

# Phase I variables set to None (not computed in this notebook)
C_tree_tau = None
C_total_tau = None
C_loop_tau = None
C_tree_residue = None
C_tree_tau_slices = None
C_total_tau_slices = None

print(f'num_params defined ({len(num_params)} entries). Phase I skipped.')

# ═══════════════════════════════════════════════════════════════
# Deferred pole / residue computation (numerical, uses num_params)
# ═══════════════════════════════════════════════════════════════
# Cell 8 left propagator_data['pole_vals'] and ['C_mats'] as None when
# the kernel image introduces rational factors (e.g., exp synaptic
# filter).  Fill them in now that num_params is available.
if propagator_data.get('pole_vals') is None and 'compute_poles_and_residues' in dir():
    _pv, _cm = compute_poles_and_residues(num_params)
    propagator_data['pole_vals'] = _pv
    propagator_data['C_mats']    = _cm
    # Keep the module-level aliases in sync so downstream cells that
    # read `pole_vals` / `C_mats` directly still work.
    pole_vals = _pv
    C_mats    = _cm
    print()
    print(f'Numerical poles ({len(pole_vals)} retarded):')
    for pk, p in enumerate(pole_vals):
        print(f'  omega_{pk+1} = {p.real:+.6f} + ({p.imag:+.6f}) i')


### 8. Time-Domain Integration

For each tree-level diagram, assign a time variable to every vertex,
pin the first external field's time to zero (stationarity), and
integrate over internal vertex times via numerical quadrature.

Each retarded propagator is decomposed as
$G^R(t) = c_\delta \cdot \delta(t) + \Theta(t) \cdot G^{\mathrm{smooth}}(t)$.
The product over edges is expanded into $2^{|E|}$ subsets;
$\delta$ factors are integrated symbolically, and the smooth
remainder is evaluated via `scipy.integrate.quad` / `nquad`.

**Convention**: `total_C(t_1, t_2, ..., t_k)` where position $i$
is the time of `external_fields[i]`.
$\tau_n = t_{n+1} - t_1$, with $t_1 = 0$ (pinned).

Surviving $\delta$ functions (e.g., $\delta(\tau_1)$) are reported
but excluded from the smooth evaluation.  $\Theta(0) = 0$ convention.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1  Evaluate tree-level contribution via Phase J
# ═══════════════════════════════════════════════════════════════
# Phase J uses explicit numerical quadrature (scipy.integrate.quad /
# nquad) on a fast_callable JIT'd integrand, with polytope bounds
# extracted from the retarded Heaviside factors.  It also enumerates
# the 2^|E| subsets of tree edges whose propagator entries have an
# instantaneous δ(t) component (the \tilde{n} × \delta n coupling in
# the MSR-JD Hawkes action): each subset's δ equalities pin
# integration variables; residual δ on external times comes back as
# a structured shot-noise spike that we insert into the τ grid below.
#
# For k=2 we evaluate on a single 1D τ grid with t_2 pinned to 0.
# For k=3 we evaluate on the two 1D slices that match the simulation
# cell's convention (cell 33):
#   slice 0: vary leaf-1\'s time, fix leaf-0 and leaf-2 at 0
#   slice 1: vary leaf-2\'s time, fix leaf-0 and leaf-1 at 0
# This matches ⟨field_a(0) · field_c(0) · field_b(τ)⟩ (slice 0) and
# ⟨field_a(0) · field_b(0) · field_c(τ)⟩ (slice 1) that the simulation
# extracts from the binned rate time series.
from msrjd.integration.time_domain import (
    compute_correction_td,
    integrate_tree_diagram,
    format_td_integral_latex,
    eval_delta_contributions_on_tau_grid,
    _loop_number_from_graph,
)

# ── Quadrature accuracy for nD polytope integration ──────────
# Default limit=200 gives high precision but 3D nquad at k=4
# takes ~20 sec per call × ~80 calls = ~25 min per point.
# Loosening to limit=50 + epsrel=1e-3 cuts to ~5 min with
# ~0.1% accuracy — plenty for theory/sim comparison.
from msrjd.integration.time_domain import final_integral as _fi
_fi.QUAD_OPTS = {'limit': 100, 'epsrel': 1e-4}



# ── Utility: compute forbidden τ values where surviving deltas fire ──
# CONVENTION: equality_a from the pipeline is indexed over FREE external
# times (excluding the pinned origin). For k=3 with origin_leaf_idx=0,
# free_ext = [t₂, t₃], so equality_a has length 2 with indices 0, 1.
# All vary_index / fixed_values in these functions use FREE-EXT indexing.

def _forbidden_tau_values(delta_contribs, vary_index_free, fixed_values_free):
    """Return a set of tau values where surviving deltas fire on a 1D slice.
    
    Parameters
    ----------
    delta_contribs : list of dict from _td_result['delta_contributions']
    vary_index_free : int
        Index into the FREE external time vector (0-based). For k=3 with
        origin at leaf 0: 0 = t₂ (τ₁), 1 = t₃ (τ₂).
    fixed_values_free : dict
        {free_ext_index: value} for all non-varying free external times.
    """
    forbidden = set()
    fixed = dict(fixed_values_free) if fixed_values_free else {}
    for dc in delta_contribs:
        eq_a = dc['equality_a']
        eq_c = dc['equality_c']
        n = len(eq_a)
        a_vary = eq_a[vary_index_free] if vary_index_free < n else 0.0
        c_eff = eq_c
        for j in range(n):
            if j != vary_index_free:
                c_eff += eq_a[j] * fixed.get(j, 0.0)
        if abs(a_vary) > 1e-15:
            tau_fire = -c_eff / a_vary
            forbidden.add(round(tau_fire, 10))
        elif abs(c_eff) < 1e-12:
            forbidden.add('ALL')
    return forbidden

def _forbidden_tau_lines_2d(delta_contribs, ax0_free, ax1_free, fixed_values_free):
    """Return list of (a0, a1, c_fixed) lines in 2D where deltas fire.
    
    Parameters use FREE-EXT indexing. Each line: a0*τ_ax0 + a1*τ_ax1 + c = 0.
    """
    lines = []
    fixed = dict(fixed_values_free) if fixed_values_free else {}
    for dc in delta_contribs:
        eq_a = dc['equality_a']
        eq_c = dc['equality_c']
        n = len(eq_a)
        a0 = eq_a[ax0_free] if ax0_free < n else 0.0
        a1 = eq_a[ax1_free] if ax1_free < n else 0.0
        c_f = eq_c
        for j in range(n):
            if j != ax0_free and j != ax1_free:
                c_f += eq_a[j] * fixed.get(j, 0.0)
        if abs(a0) > 1e-15 or abs(a1) > 1e-15:
            lines.append((a0, a1, c_f))
    return lines


if k == 1:
    # k=1: stationary 1-point fluctuation function.  At MF saddle the
    # tree-level contribution is 0 (phi_0 = n* cancels the tadpole),
    # so anything we see here is a pure LOOP correction from the
    # nonlinear vertices (phi'' for the quadratic model, exp(n~) for
    # the Poisson nonlinearity).  To show the tree vs loop split
    # explicitly in the cell-32 plot we evaluate TWO separate sums:
    #   _td_result       -- every diagram (tree + 1-loop + ...)
    #   _td_tree_result  -- tree-only subset
    _origin_idx = 0
    _td_result = compute_correction_td(
        typed_diagrams=all_unique,
        prefactors=diagram_prefactors,
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=_origin_idx,
        external_fields=external_fields,
    )
    _tree_only_idx = [i for i, td in enumerate(all_unique)
                      if _loop_number_from_graph(td) == 0]
    _td_tree_result = compute_correction_td(
        typed_diagrams=[all_unique[i] for i in _tree_only_idx],
        prefactors=[diagram_prefactors[i] for i in _tree_only_idx],
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=_origin_idx,
        external_fields=external_fields,
    )

    _n_tree_full  = sum(1 for g in _td_result['groups']
                        if g['handled_by'] == 'tree_evaluator')
    _n_loop_full  = sum(1 for g in _td_result['groups']
                        if g['handled_by'] == 'loop_evaluator')
    _n_skipped    = len(_td_result['skipped_kernel_ids'])
    print(f'Phase J (k=1): evaluated {_n_tree_full} tree + {_n_loop_full} '
          f'1-loop diagrams, skipped {_n_skipped}.')

    # Tau grid stays None for k=1 (no free external time).
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None

elif k == 2:
    # Canonical convention: pin t_1 (position 0) → 0; leave t_2 free = τ.
    # This matches the pipeline's documented convention in
    # msrjd/integration/time_domain/pipeline.py and the simulation's
    # cross-correlation convention ⟨δX_0(t) · δX_1(t + τ)⟩ from
    # compute_kpoint_slice.  (Historical note: _origin_idx was 1 here
    # for legacy reasons, which produced a τ → -τ mirror image relative
    # to sim.  Fixed 2026-04-15.)
    #
    # CRITICAL: `external_fields=external_fields` MUST be passed to
    # compute_correction_td.  Without it, integrate_tree_diagram falls
    # back to an identity mapping based on the enumeration's leaf
    # order, which (for mixed fields like [dn_1, dv_2]) can be the
    # REVERSE of external_fields — producing a τ → -τ mirror image of
    # ⟨δn_1(0) · δv_2(τ)⟩.  This was the second half of the 2026-04-17
    # mirror-image fix.
    _origin_idx = 0
    _td_result = compute_correction_td(
        typed_diagrams=all_unique,
        prefactors=diagram_prefactors,
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=_origin_idx,
        external_fields=external_fields,
    )

    n_tree_groups = sum(
        1 for g in _td_result['groups'] if g['handled_by'] == 'tree_evaluator'
    )
    n_skipped = len(_td_result['skipped_kernel_ids'])
    n_delta_total = len(_td_result['delta_contributions'])
    print(f'Phase J (k={k}): evaluated {n_tree_groups} tree kernel group(s), '
          f'skipped {n_skipped} loop kernel group(s), '
          f'{n_delta_total} shot-noise δ spike(s) produced.')
    for g in _td_result['groups']:
        extra = ''
        if 'n_delta_contributions' in g and g['n_delta_contributions']:
            extra = f"  δ-spikes={g['n_delta_contributions']}"
        print(f"   kernel ell={g['loop_number']}  "
              f"n_diagrams={g['n_diagrams']}  "
              f"handled_by={g['handled_by']}  "
              f"representation={g['representation']}  {g['reason']}{extra}")

    # Display the time-domain integral for each tree kernel group.
    _ext_syms = [SR.var(f't{j+1}_J') for j in range(k)]
    for _gi, _g in enumerate(_td_result['groups']):
        if _g['handled_by'] != 'tree_evaluator':
            continue
        _diag_idx = _g.get('kernel_id')
        if _diag_idx is None or _diag_idx >= len(all_unique):
            continue
        _td_diag = all_unique[_diag_idx]
        _td_pf = diagram_prefactors[_diag_idx]
        _per_group_result = integrate_tree_diagram(
            typed_diagram=_td_diag,
            propagator_data=propagator_data,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            num_params=num_params,
            origin_leaf_idx=_origin_idx,
            external_fields=external_fields,
        )
        _label = f'Diagram {_diag_idx}  (ell={_g["loop_number"]})'
        _latex_str = format_td_integral_latex(
            _per_group_result,
            typed_diagram=_td_diag,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            label=_label,
        )
        display(Math(_latex_str))

    # Evaluate on a single 1D τ grid (t_2 pinned)
    # (Numerical τ-grid evaluation is in cell 8.1c below.)

elif k >= 3:
    # k=3: no pinning — Phase J returns C(t_1, t_2, t_3) and we evaluate
    # slices matching the simulation conventions.
    #
    # For each value in TAU_FIXED_LIST we evaluate TWO slices:
    #   slice s=0:  vary leaf 1 time → C(0, τ, τ_fixed)
    #   slice s=1:  vary leaf 2 time → C(0, τ_fixed, τ)
    # τ_fixed = 0 recovers the two canonical "other τ pinned to origin"
    # slices; nonzero τ_fixed lets us peek at off-diagonal cuts of the
    # 2D cumulant surface.
    TAU_FIXED_LIST = [1.0, -1.0]   # matches test points (±2, ±1)

    _td_result = compute_correction_td(
        typed_diagrams=all_unique,
        prefactors=diagram_prefactors,
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=None,
        external_fields=external_fields,
    )

    n_tree_groups = sum(
        1 for g in _td_result['groups'] if g['handled_by'] == 'tree_evaluator'
    )
    n_skipped = len(_td_result['skipped_kernel_ids'])
    n_delta_total = len(_td_result['delta_contributions'])
    print(f'Phase J (k={k}): evaluated {n_tree_groups} tree kernel group(s), '
          f'skipped {n_skipped} loop kernel group(s), '
          f'{n_delta_total} shot-noise δ spike(s) produced.')
    for g in _td_result['groups']:
        extra = ''
        if 'n_delta_contributions' in g and g['n_delta_contributions']:
            extra = f"  δ-spikes={g['n_delta_contributions']}"
        print(f"   kernel ell={g['loop_number']}  "
              f"n_diagrams={g['n_diagrams']}  "
              f"handled_by={g['handled_by']}  "
              f"representation={g['representation']}  {g['reason']}{extra}")

    # Display the time-domain integral for each tree diagram.
    _ext_syms = [SR.var(f't{j+1}_J') for j in range(k)]
    for _gi, _g in enumerate(_td_result['groups']):
        if _g['handled_by'] != 'tree_evaluator':
            continue
        _diag_idx = _g.get('kernel_id')
        if _diag_idx is None or _diag_idx >= len(all_unique):
            continue
        _td_diag = all_unique[_diag_idx]
        _td_pf = diagram_prefactors[_diag_idx]
        _per_group_result = integrate_tree_diagram(
            typed_diagram=_td_diag,
            propagator_data=propagator_data,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            num_params=num_params,
            origin_leaf_idx=None,
            external_fields=external_fields,
        )
        _label = f'Diagram {_diag_idx}  (ell={_g["loop_number"]})'
        _latex_str = format_td_integral_latex(
            _per_group_result,
            typed_diagram=_td_diag,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            label=_label,
        )
        display(Math(_latex_str))

    # Phase J callable signature is (t_1, t_2, t_3).
    # For each (s, τ_fixed) combo:
    #   s = 0:  vary leaf 1 → C(0, τ, τ_fixed)   (τ_other = t_3 fixed)
    #   s = 1:  vary leaf 2 → C(0, τ_fixed, τ)   (τ_other = t_2 fixed)
    # τ_fixed = 0 gives the canonical slices comparable with the
    # simulation's `τ_other = 0` plots; τ_fixed = +5 probes an
    # off-diagonal cut through the 3-point surface.
    # (Numerical τ-grid evaluation is in cell 8.1c below.)
else:
    print(f'Phase J prototype currently supports k >= 2; '
          f'current k={k}.  Nothing to do.')
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1b  Symbolic δ / smooth propagator decomposition (5 steps)
# ═══════════════════════════════════════════════════════════════
# For each tree diagram, show the algebraic steps that integrate out
# the δ(t) component of the retarded propagator:
#
#   G^R_{p,r}(Δt) = c_δ · δ(Δt) + Θ(Δt) · G^{sm}_{p,r}(Δt)
#
# Step 1: Full integral with G^R
# Step 2: Expand G^R into δ + Θ·G^sm per edge
# Step 3: Distribute → 2^|E_δ| terms
# Step 4: Integrate out δ (sifting property)
# Step 5: Substitute stationary variables τ_j = t_{j+1} − t_1

from msrjd.integration.time_domain import build_G_t_matrix, G_t_delta_coeff
from msrjd.integration.time_domain.final_integral import (
    _lookup_prop_indices, _loop_number_from_graph,
)
from IPython.display import display, Math

# ── Toggle: suppress symbolic LaTeX output (Steps 1-5) ──────
# Set SHOW_SYMBOLIC = False to skip the display(Math(...)) calls
# for Steps 1-5.  Much faster for k=4 where the LaTeX rendering
# is lengthy.
# Set SHOW_NUMERIC  = False to skip the per-point numerical check
# output.  The underlying computation still runs so that
# hand_eval_results is populated for downstream cells (8.1b', 9-quick).
SHOW_SYMBOLIC = False
SHOW_NUMERIC  = False
SKIP_CELL     = True    # True → skip ALL computation (fast).
                        # Cell 31 only needs _td_result from cell 25,
                        # not hand_eval_results from here.

# ── Helper: field LaTeX name from matrix index ──────────────
_pop_list = list(HAWKES_MODEL['index_sets']['pop'])

def _field_lx(field_specs, matrix_idx):
    """Map a 0-based propagator matrix index to a LaTeX field label."""
    offset = 0
    for fspec in field_specs:
        lx = fspec.get('latex', fspec['name'])
        if fspec.get('indexed', True):
            n_sub = len(_pop_list)
            if matrix_idx < offset + n_sub:
                pop_i = _pop_list[matrix_idx - offset]
                return lx + '_{' + str(pop_i + 1) + '}'
            offset += n_sub
        else:
            if matrix_idx == offset:
                return lx
            offset += 1
    return str(matrix_idx)

_resp_specs = HAWKES_MODEL['response_fields']
_phys_specs = HAWKES_MODEL['physical_fields']

def _sub(pi, ri):
    """Propagator subscript {phys, resp}."""
    return '{' + _field_lx(_phys_specs, pi) + r',\,' + _field_lx(_resp_specs, ri) + '}'

def _GR(pi, ri, dt_lx):
    return r'G^{R}_{' + _sub(pi, ri) + r'}(' + dt_lx + r')'

def _Gsm(pi, ri, dt_lx):
    return r'G^{\mathrm{sm}}_{' + _sub(pi, ri) + r'}(' + dt_lx + r')'

def _theta(dt_lx):
    return r'\Theta(' + dt_lx + r')'

def _delta_fn(dt_lx):
    return r'\delta(' + dt_lx + r')'

def _c_lx(c_val):
    """Delta coefficient as LaTeX (suppressed if 1)."""
    if abs(c_val - 1.0) < 1e-12:
        return ''
    if abs(c_val - (-1.0)) < 1e-12:
        return '-'
    return f'{c_val:.4g}' + r'\,'


# ── Helper: gather edge info ────────────────────────────────
def _gather_edges(td, G_obj, vertex_time):
    D = td.prediagram[0]
    info = []
    for (u, v, lbl) in D.edges():
        ri, pi = _lookup_prop_indices(td, (u, v, lbl))
        dt = vertex_time[v] - vertex_time[u]
        c_d = G_t_delta_coeff(G_obj, pi, ri)
        has_delta = abs(complex(c_d)) > 1e-15
        c_val = complex(c_d).real if has_delta else 0.0
        info.append(dict(u=u, v=v, ri=ri, pi=pi, dt=dt,
                         c_delta=c_val, has_delta=has_delta))
    return info


# ── Helper: iterate valid subsets ───────────────────────────
def _valid_subsets(edges):
    """Yield (bits, d_chosen, s_chosen) for each valid 2^|E_δ| subset."""
    n_e = len(edges)
    for bits in range(2**n_e):
        d_chosen = [i for i in range(n_e) if (bits >> i) & 1]
        s_chosen = [i for i in range(n_e) if not ((bits >> i) & 1)]
        if any(not edges[i]['has_delta'] for i in d_chosen):
            continue
        yield bits, d_chosen, s_chosen


# ── Helper: solve delta constraints ─────────────────────────
def _solve_deltas(edges, d_chosen, int_vars):
    """Solve δ constraints; return (subs, remaining, residuals, coeff)."""
    subs = {}
    remaining = list(int_vars)
    residuals = []
    coeff = SR(1)
    for i in d_chosen:
        coeff *= SR(edges[i]['c_delta'])
        dt_sub = SR(edges[i]['dt']).subs(subs)
        solved = False
        for iv in list(remaining):
            if iv in dt_sub.variables():
                sol = solve(dt_sub == 0, iv, solution_dict=True)
                if sol:
                    subs[iv] = sol[0][iv]
                    remaining.remove(iv)
                    solved = True
                    break
        if not solved:
            res_expr = dt_sub.subs(subs)
            if not res_expr.is_zero():
                residuals.append(res_expr)
    return subs, remaining, residuals, coeff


# ── Helper: build smooth factors LaTeX ──────────────────────
def _smooth_factors_lx(edges, s_chosen, subs):
    """LaTeX for the product of smooth propagator factors after δ subs."""
    parts = []
    for i in s_chosen:
        e = edges[i]
        dt_sub = SR(e['dt']).subs(subs)
        dt_lx = latex(dt_sub)
        parts.append(_theta(dt_lx) + r'\,' + _Gsm(e['pi'], e['ri'], dt_lx))
    return parts


# ── Step builders ───────────────────────────────────────────

def _int_measure(variables):
    return ''.join(r'\int_{-\infty}^{\infty}\!d' + latex(s) + r'\;'
                   for s in variables)


def build_step1(edges, int_vars, pf_lx):
    prod = r'\,'.join(_GR(e['pi'], e['ri'], latex(e['dt'])) for e in edges)
    return pf_lx + r'\,' + _int_measure(int_vars) + prod


def build_step2(edges, int_vars, pf_lx):
    factors = []
    for e in edges:
        dt_lx = latex(e['dt'])
        if e['has_delta']:
            c = _c_lx(e['c_delta'])
            factors.append(
                r'\bigl(' + c + _delta_fn(dt_lx) + r' + '
                + _theta(dt_lx) + r'\,' + _Gsm(e['pi'], e['ri'], dt_lx)
                + r'\bigr)')
        else:
            factors.append(
                _theta(dt_lx) + r'\,' + _Gsm(e['pi'], e['ri'], dt_lx))
    return pf_lx + r'\,' + _int_measure(int_vars) + r'\,'.join(factors)


def build_step3(edges, int_vars, pf_lx):
    """Returns list of (term_latex,) strings for each subset."""
    lines = []
    for bits, d_chosen, s_chosen in _valid_subsets(edges):
        parts = []
        for i in d_chosen:
            e = edges[i]
            parts.append(_c_lx(e['c_delta']) + _delta_fn(latex(e['dt'])))
        for i in s_chosen:
            e = edges[i]
            dt_lx = latex(e['dt'])
            parts.append(_theta(dt_lx) + r'\,' + _Gsm(e['pi'], e['ri'], dt_lx))
        prod = r'\,'.join(parts) if parts else '1'
        lines.append(_int_measure(int_vars) + prod)
    return lines


def build_step4(edges, int_vars, ext_syms, pf_lx):
    """Returns list of dict(latex=..., is_delta=..., delta_arg_lx=...)."""
    results = []
    for bits, d_chosen, s_chosen in _valid_subsets(edges):
        subs, remaining, residuals, coeff = _solve_deltas(
            edges, d_chosen, int_vars)
        dc_lx = _c_lx(float(SR(coeff)))
        smooth = _smooth_factors_lx(edges, s_chosen, subs)
        is_delta = len(residuals) > 0
        delta_arg_lx = None

        if remaining:
            body = r'\,'.join(smooth) if smooth else '1'
            result_lx = dc_lx + _int_measure(remaining) + body
        elif residuals:
            res_parts = [_delta_fn(latex(r)) for r in residuals]
            body = r'\,'.join(res_parts + smooth)
            result_lx = dc_lx + body
            delta_arg_lx = latex(residuals[0])
            is_delta = True
        else:
            body = r'\,'.join(smooth) if smooth else '1'
            result_lx = dc_lx + body

        results.append(dict(latex=result_lx, is_delta=is_delta,
                            delta_arg_lx=delta_arg_lx))
    return results


def build_step5_subs(ext_syms, int_vars, origin_idx=0):
    """Build the τ / u substitution dicts and header LaTeX.

    Convention:  τ_j = t_{j+1} - t_origin,  u = s - t_origin.
    We substitute  t_{j+1} → t_origin + τ_j  and  s → t_origin + u.
    The origin time t_origin cancels algebraically in every time
    difference, so we never need to set t_origin = 0 explicitly.
    """
    t_origin = ext_syms[origin_idx]
    tau_subs = {}   # do NOT include t_origin → 0
    tau_syms = []
    free_idx = 0
    for j, t_j in enumerate(ext_syms):
        if j == origin_idx:
            continue
        free_idx += 1
        tau = SR.var(f'tau_{free_idx}',
                     latex_name=r'\tau_{' + str(free_idx) + '}')
        tau_subs[t_j] = t_origin + tau   # t_j → t_origin + τ
        tau_syms.append(tau)

    u_subs = {}
    u_syms = []
    for iv_idx, iv in enumerate(int_vars):
        u = SR.var('u' if len(int_vars) == 1
                   else f'u_{iv_idx + 1}')
        u_subs[iv] = t_origin + u   # s → t_origin + u
        u_syms.append(u)

    all_subs = {**tau_subs, **u_subs}

    # Header line: show the *definitions*, not a substitution on t_origin
    defs = []
    free_idx = 0
    for j, t_j in enumerate(ext_syms):
        if j == origin_idx:
            continue
        free_idx += 1
        tau_sym = tau_syms[free_idx - 1]
        defs.append(latex(tau_sym) + ' \\equiv ' + latex(t_j)
                    + ' - ' + latex(t_origin))
    for iv, u in zip(int_vars, u_syms):
        defs.append(latex(u) + ' \\equiv ' + latex(iv)
                    + ' - ' + latex(t_origin))

    header_lx = r'\text{Let }\; ' + r',\;\; '.join(defs)
    return all_subs, tau_syms, u_syms, header_lx


def build_step5_term(edges, int_vars, d_chosen, s_chosen, all_subs, u_syms, ext_syms, origin_idx=0):
    """Rebuild one term with stationary substitutions."""
    subs, remaining, residuals, coeff = _solve_deltas(
        edges, d_chosen, int_vars)
    dc_lx = _c_lx(float(SR(coeff)))

    # Apply stationary subs on top of delta subs
    smooth = []
    for i in s_chosen:
        e = edges[i]
        dt_sub = SR(e['dt']).subs(subs).subs(all_subs)
        dt_lx = latex(dt_sub)
        smooth.append(_theta(dt_lx) + r'\,' + _Gsm(e['pi'], e['ri'], dt_lx))

    if remaining:
        # Map remaining int vars through all_subs.
        # After the stationary substitution s → t_origin + u, the
        # integration variable is u (not s).  Extract the u symbol.
        mapped_vars = []
        for s in remaining:
            mv = all_subs.get(s, s)
            # mv = t_origin + u; extract the u symbol
            if hasattr(mv, 'operands') and len(mv.operands()) == 2:
                for op in mv.operands():
                    if op not in ext_syms:
                        mapped_vars.append(op)
                        break
                else:
                    mapped_vars.append(mv)
            else:
                mapped_vars.append(mv)
        meas = _int_measure(mapped_vars)
        body = r'\,'.join(smooth) if smooth else '1'
        return dc_lx + meas + body
    elif residuals:
        res_parts = [_delta_fn(latex(SR(r).subs(all_subs)))
                     for r in residuals]
        body = r'\,'.join(res_parts + smooth)
        return dc_lx + body
    else:
        body = r'\,'.join(smooth) if smooth else '1'
        return dc_lx + body




# ── Per-term numerical evaluation ───────────────────────────
from msrjd.integration.time_domain import G_t_entry
from sage.all import fast_callable, CDF, heaviside
import numpy as np
try:
    from scipy.integrate import quad, nquad
except ImportError:
    quad = None

def _build_term_callable(edges, int_vars, d_chosen, s_chosen,
                         G_obj, all_subs, u_syms, ext_syms,
                         pf_num, origin_idx=0):
    """Build a numerical callable for one 2^|E| subset term.

    Returns (fn, kind) where:
      fn(*tau_vals) -> float   (tau_vals are the free external times)
      kind = 'smooth_integral' | 'smooth_direct' | 'delta'

    The retarded Heaviside Θ(Δt) is NOT baked into the symbolic
    expression (fast_callable cannot compile it).  Instead, we
    collect the time-difference arguments and enforce Θ numerically.
    """
    subs_d, remaining, residuals, coeff = _solve_deltas(
        edges, d_chosen, int_vars)
    coeff_num = float(SR(coeff))

    t_origin = ext_syms[origin_idx]

    if residuals:
        return None, 'delta'

    # Build symbolic product WITHOUT Heaviside, but collect the
    # Heaviside arguments for numerical enforcement.
    smooth_product = SR(1)
    theta_args = []          # symbolic expressions that must be >= 0
    for i in s_chosen:
        e = edges[i]
        dt_sub = SR(e['dt']).subs(subs_d).subs(all_subs)
        entry = G_t_entry(G_obj, e['pi'], e['ri'], dt_sub,
                          include_heaviside=False)
        smooth_product *= entry
        theta_args.append(dt_sub)

    # Identify free variables: tau symbols and (if any) u variables
    tau_syms_list = []
    for j, t_j in enumerate(ext_syms):
        if j == origin_idx:
            continue
        tau_syms_list.append(all_subs[t_j] - t_origin)

    if remaining:
        u_vars = []
        for s in remaining:
            mv = all_subs.get(s, s)
            if hasattr(mv, 'operands') and len(mv.operands()) == 2:
                for op in mv.operands():
                    if op not in ext_syms:
                        u_vars.append(op)
                        break
                else:
                    u_vars.append(mv)
            else:
                u_vars.append(mv)

        all_vars = tau_syms_list + u_vars
        try:
            fc = fast_callable(smooth_product, vars=all_vars, domain=CDF)
            theta_fcs = [fast_callable(a, vars=all_vars, domain=CDF)
                         for a in theta_args]
        except Exception as exc:
            return None, 'failed'

        if len(u_vars) == 1:
            # Extract polytope bounds analytically from theta args.
            # Each theta is linear: a_u * u + (function of tau) >= 0.
            # The intersection of these half-lines is a single interval.
            u_var = u_vars[0]
            theta_a_u = []   # coefficient of u in each theta
            theta_const_fcs = []  # callable for theta evaluated at u=0
            for ta in theta_args:
                ta_sr = SR(ta)
                a = float(ta_sr.coefficient(u_var))
                const_part = ta_sr - a * u_var
                # const_part depends only on tau symbols
                try:
                    const_fc = fast_callable(const_part, vars=tau_syms_list,
                                             domain=CDF)
                except Exception:
                    const_fc = None
                theta_a_u.append(a)
                theta_const_fcs.append(const_fc)

            def _fn(*tau_vals, _fc=fc, _a_us=theta_a_u,
                    _const_fcs=theta_const_fcs, _c=coeff_num, _pf=pf_num):
                # Compute bounds [L, U] from polytope at this tau
                L, U = -200.0, 200.0
                for a_u, const_fc in zip(_a_us, _const_fcs):
                    if const_fc is None:
                        continue
                    b = float(complex(const_fc(*tau_vals)).real)
                    # Constraint: a_u * u + b >= 0
                    if abs(a_u) < 1e-15:
                        # Pure tau constraint
                        if b < 0:
                            return 0.0  # infeasible
                        continue
                    bound = -b / a_u
                    if a_u > 0:
                        # u >= bound
                        if bound > L:
                            L = bound
                    else:
                        # u <= bound
                        if bound < U:
                            U = bound
                if L >= U:
                    return 0.0
                # Clip to safety range
                L = max(L, -500.0)
                U = min(U, 500.0)

                def _integrand(u_val):
                    args = list(tau_vals) + [u_val]
                    return complex(_fc(*args)).real
                val, _ = quad(_integrand, L, U, limit=200)
                return _pf * _c * val
            return _fn, 'smooth_integral'
        elif len(u_vars) == 2:
            # Extract polytope bounds analytically.
            # Each theta is linear: a_u1*u1 + a_u2*u2 + b(tau) >= 0
            u1_var, u2_var = u_vars[0], u_vars[1]
            theta_a_u1 = []
            theta_a_u2 = []
            theta_const_fcs = []
            for ta in theta_args:
                ta_sr = SR(ta)
                a1 = float(ta_sr.coefficient(u1_var))
                a2 = float(ta_sr.coefficient(u2_var))
                const_part = ta_sr - a1 * u1_var - a2 * u2_var
                try:
                    const_fc = fast_callable(const_part, vars=tau_syms_list,
                                             domain=CDF)
                except Exception:
                    const_fc = None
                theta_a_u1.append(a1)
                theta_a_u2.append(a2)
                theta_const_fcs.append(const_fc)

            def _fn(*tau_vals, _fc=fc, _a1s=theta_a_u1, _a2s=theta_a_u2,
                    _const_fcs=theta_const_fcs, _c=coeff_num, _pf=pf_num):
                # Evaluate constants at this tau
                bs = []
                for const_fc in _const_fcs:
                    if const_fc is None:
                        bs.append(None)
                    else:
                        bs.append(float(complex(const_fc(*tau_vals)).real))

                # Outer variable u1: find bounds from theta with a_u2=0
                L1, U1 = -200.0, 200.0
                for a1, a2, b in zip(_a1s, _a2s, bs):
                    if b is None:
                        continue
                    if abs(a2) >= 1e-15:
                        continue  # depends on u2 too — handle in inner
                    if abs(a1) < 1e-15:
                        if b < 0:
                            return 0.0  # infeasible
                        continue
                    bound = -b / a1
                    if a1 > 0:
                        if bound > L1:
                            L1 = bound
                    else:
                        if bound < U1:
                            U1 = bound
                if L1 >= U1:
                    return 0.0
                L1 = max(L1, -500.0)
                U1 = min(U1, 500.0)

                # Inner u2 bounds: depend on u1 and tau
                def bounds_u2(u1_val):
                    L2, U2 = -200.0, 200.0
                    for a1, a2, b in zip(_a1s, _a2s, bs):
                        if b is None:
                            continue
                        if abs(a2) < 1e-15:
                            continue  # already handled in outer
                        b_eff = b + a1 * u1_val
                        bound = -b_eff / a2
                        if a2 > 0:
                            if bound > L2:
                                L2 = bound
                        else:
                            if bound < U2:
                                U2 = bound
                    if L2 >= U2:
                        return (0.0, 0.0)
                    L2 = max(L2, -500.0)
                    U2 = min(U2, 500.0)
                    return (L2, U2)

                def _integrand(u2, u1):
                    args = list(tau_vals) + [u1, u2]
                    return complex(_fc(*args)).real

                val, _ = nquad(_integrand,
                               [bounds_u2, (L1, U1)],
                               opts={'limit': 100})
                return _pf * _c * val
            return _fn, 'smooth_integral_2d'
        else:
            return None, 'multi_integral'
    else:
        # Direct evaluation (no integration variables remain)
        try:
            fc = fast_callable(smooth_product, vars=tau_syms_list, domain=CDF)
            theta_fcs = [fast_callable(a, vars=tau_syms_list, domain=CDF)
                         for a in theta_args]
        except Exception:
            return None, 'failed'

        def _fn(*tau_vals, _fc=fc, _thetas=theta_fcs,
                _c=coeff_num, _pf=pf_num):
            for th in _thetas:
                if complex(th(*tau_vals)).real < 0:
                    return 0.0
            return _pf * _c * complex(_fc(*tau_vals)).real
        return _fn, 'smooth_direct'

# ═══════════════════════════════════════════════════════════════
# Main display loop
# ═══════════════════════════════════════════════════════════════
# Results dict: populated by the loop below, reused by 8.1b' and 9-quick.
# hand_eval_results[gi] = {(tau1, tau2): {'terms': [...], 'total': float, 'pipeline': float}}
hand_eval_results = {}

hand_eval_results = {}  # populated below if SKIP_CELL is False

if k >= 2 and not SKIP_CELL:
    _t_sym = SR.var('_t_decomp_')
    _G_obj = build_G_t_matrix(propagator_data, _t_sym, num_params=num_params)

    for gi, td in enumerate(all_unique):
        if _loop_number_from_graph(td) != 0:
            continue

        D = td.prediagram[0]
        leaves = list(td.prediagram[2])
        leaf_set = set(leaves)
        internal = [v for v in D.vertices() if v not in leaf_set]

        # Assign symbolic times using CANONICAL field remapping
        # (matches pipeline's integrate_tree_diagram convention:
        #  position i in ext_syms = time of external_fields[i])
        ext_syms = [SR.var(f't_{j+1}', latex_name=f't_{{{j+1}}}')
                    for j in range(k)]
        vertex_time = {}
        _used_canon = set()
        _leaf_to_canon = {}
        for j, lf in enumerate(leaves):
            field = td.external_legs.get(lf)
            for cp in range(len(external_fields)):
                if cp not in _used_canon and external_fields[cp] == field:
                    _leaf_to_canon[j] = cp
                    _used_canon.add(cp)
                    break
            else:
                _leaf_to_canon[j] = j  # fallback
        for j, lf in enumerate(leaves):
            vertex_time[lf] = ext_syms[_leaf_to_canon[j]]
        int_vars = []
        for v in internal:
            s_v = SR.var(f's_{v}', latex_name=f's_{{{v}}}')
            vertex_time[v] = s_v
            int_vars.append(s_v)

        edges = _gather_edges(td, _G_obj, vertex_time)
        n_delta = sum(1 for e in edges if e['has_delta'])
        pf_lx = latex(diagram_prefactors[gi])

        # ── Header ──
        if SHOW_SYMBOLIC:
            display(Math(
            r'\boxed{\textbf{Diagram\;' + str(gi) + r'}'
            + r'\quad \text{prefactor} = ' + pf_lx
            + r',\quad ' + str(len(edges)) + r'\text{ edges, }'
            + str(n_delta) + r'\text{ with nonzero } \delta'
            + r'}'))

        if n_delta == 0:
            # All edges purely smooth — no δ decomposition needed.
            # The 2^|E_δ| sum reduces to a single smooth integral,
            # which _valid_subsets yields as the only subset.  Display
            # Step 1 for context, but do NOT `continue`: we still need
            # to run the per-point numerical evaluation below so that
            # hand_eval_results[gi] is populated for this diagram.
            if SHOW_SYMBOLIC:
                display(Math(r'\text{All edges purely smooth --- '
                         r'no }\delta\text{ decomposition needed.}'))
                display(Math(build_step1(edges, int_vars, pf_lx)))
            # ── intentionally no `continue` here ──
        else:
            # ── Step 1: Full integral ──
            if SHOW_SYMBOLIC:
                display(Math(r'\textbf{Step 1}\text{ --- full integral:}'))
            if SHOW_SYMBOLIC:
                display(Math(build_step1(edges, int_vars, pf_lx)))

            # ── Step 2: Expand G^R ──
            if SHOW_SYMBOLIC:
                display(Math(r'\textbf{Step 2}\text{ --- expand } G^R = '
                         r'c_\delta\,\delta + \Theta\,G^{\mathrm{sm}}'
                         r'\text{:}'))
            if SHOW_SYMBOLIC:
                display(Math(build_step2(edges, int_vars, pf_lx)))

            # ── Step 3: Distribute ──
            s3_lines = build_step3(edges, int_vars, pf_lx)
            if SHOW_SYMBOLIC:
                display(Math(r'\textbf{Step 3}\text{ --- distribute ('
                         + str(len(s3_lines))
                         + r' terms):}'))
            s3_parts = []
            for idx, line in enumerate(s3_lines):
                pfx = r'&= ' if idx == 0 else r'&\quad +\; '
                s3_parts.append(pfx + pf_lx + r'\,\Bigl[' + line + r'\Bigr]')
            if SHOW_SYMBOLIC:
                display(Math(r'\begin{aligned} '
                         + (r' \\[6pt] ').join(s3_parts)
                         + r' \end{aligned}'))

            # ── Step 4: Integrate out δ ──
            s4_results = build_step4(edges, int_vars, ext_syms, pf_lx)
            if SHOW_SYMBOLIC:
                display(Math(r'\textbf{Step 4}\text{ --- integrate out }\delta'
                         r'\text{ (sifting property):}'))
            s4_parts = []
            for idx, res in enumerate(s4_results):
                pfx = r'&= ' if idx == 0 else r'&\quad +\; '
                annot = r'\quad\text{(shot noise)}' if res['is_delta'] else ''
                s4_parts.append(pfx + pf_lx + r'\,\Bigl[' + res['latex']
                                + r'\Bigr]' + annot)
            if SHOW_SYMBOLIC:
                display(Math(r'\begin{aligned} '
                         + (r' \\[6pt] ').join(s4_parts)
                         + r' \end{aligned}'))


        # ── Step 5: Stationary variables ──
        all_subs, tau_syms, u_syms, header_lx = build_step5_subs(
            ext_syms, int_vars, origin_idx=0)
        if SHOW_SYMBOLIC:
            display(Math(r'\textbf{Step 5}\text{ --- stationary variables:}'))
        if SHOW_SYMBOLIC:
            display(Math(header_lx))

        s5_parts = []
        term_idx = 0
        for bits, d_chosen, s_chosen in _valid_subsets(edges):
            term_lx = build_step5_term(
                edges, int_vars, d_chosen, s_chosen, all_subs, u_syms,
                ext_syms, origin_idx=0)
            if term_lx is None:
                continue
            pfx = r'&= ' if term_idx == 0 else r'&\quad +\; '
            s5_parts.append(pfx + pf_lx + r'\,\Bigl[' + term_lx + r'\Bigr]')
            term_idx += 1
        if SHOW_SYMBOLIC:
            display(Math(r'\begin{aligned} '
                     + (r' \\[6pt] ').join(s5_parts)
                     + r' \end{aligned}'))

        # ── Numerical evaluation at test points ──
        _test_pts = ([(3.0, 2.0, 1.0), (3.0, 2.0, -1.0), (3.0, -2.0, 1.0), (3.0, -2.0, -1.0), (-3.0, 2.0, 1.0), (-3.0, 2.0, -1.0), (-3.0, -2.0, 1.0), (-3.0, -2.0, -1.0)] if k == 4 else ([(-20.0, 1.0), (-20.0, -1.0), (-20.0, 5.0), (-20.0, -5.0)] if k == 3 else [(-20.0,), (-5.0,)]))
        _test_labels = ([r'\\tau_1\\!=\\!3,\\,\\tau_2\\!=\\!2,\\,\\tau_3\\!=\\!1',
                         r'\\tau_1\\!=\\!3,\\,\\tau_2\\!=\\!2,\\,\\tau_3\\!=\\!{-1}',
                         r'\\tau_1\\!=\\!3,\\,\\tau_2\\!=\\!{-2},\\,\\tau_3\\!=\\!1',
                         r'\\tau_1\\!=\\!3,\\,\\tau_2\\!=\\!{-2},\\,\\tau_3\\!=\\!{-1}',
                         r'\\tau_1\\!=\\!{-3},\\,\\tau_2\\!=\\!2,\\,\\tau_3\\!=\\!1',
                         r'\\tau_1\\!=\\!{-3},\\,\\tau_2\\!=\\!2,\\,\\tau_3\\!=\\!{-1}',
                         r'\\tau_1\\!=\\!{-3},\\,\\tau_2\\!=\\!{-2},\\,\\tau_3\\!=\\!1',
                         r'\\tau_1\\!=\\!{-3},\\,\\tau_2\\!=\\!{-2},\\,\\tau_3\\!=\\!{-1}']
                        if k == 4 else
                        ([r'\tau_1\!=\!{-20},\,\tau_2\!=\!1',
                          r'\tau_1\!=\!{-20},\,\tau_2\!=\!{-1}',
                          r'\tau_1\!=\!{-20},\,\tau_2\!=\!5',
                          r'\tau_1\!=\!{-20},\,\tau_2\!=\!{-5}']
                         if k == 3 else [r'\tau\!=\!{-20}', r'\tau\!=\!{-5}']))
        _pf_num = float(SR(diagram_prefactors[gi]).subs(num_params))

        _eval_lines = [r'\textbf{Numerical check:}']
        for _pt, _lbl in zip(_test_pts, _test_labels):
            _term_vals = []
            _total_hand = 0.0
            _term_idx2 = 0
            for _bits, _dc, _sc in _valid_subsets(edges):
                _fn, _kind = _build_term_callable(
                    edges, int_vars, _dc, _sc,
                    _G_obj, all_subs, u_syms, ext_syms,
                    pf_num=_pf_num, origin_idx=0)
                if _fn is not None:
                    _v = _fn(*_pt)
                    pass  # _v already includes prefactor × coeff
                    _term_vals.append((_term_idx2, _v, _kind))
                else:
                    _term_vals.append((_term_idx2, None, _kind))
                _term_idx2 += 1

            # Recompute properly
            _total_hand = sum(tv[1] for tv in _term_vals
                              if tv[1] is not None)
            _parts = [f'T_{tv[0]}={tv[1]:.6e}'
                      if tv[1] is not None else f'T_{tv[0]}=\\delta'
                      for tv in _term_vals]
            _sum_str = f'{_total_hand:.6e}'

            # Pipeline value
            _pipeline_val = None
            for _g_info in _td_result['groups']:
                if _g_info.get('kernel_id') == gi and _g_info.get('contribution') is not None:
                    _pv = _g_info['contribution'](0.0, *_pt)
                    _pipeline_val = float(complex(_pv).real)
                    break

            # Store for reuse by downstream cells
            if gi not in hand_eval_results:
                hand_eval_results[gi] = {}
            hand_eval_results[gi][_pt] = {
                'terms': [{'idx': tv[0], 'val': tv[1], 'kind': tv[2]}
                          for tv in _term_vals],
                'total': _total_hand,
                'pipeline': _pipeline_val,
            }

            _pv_str = f'{_pipeline_val:.6e}' if _pipeline_val is not None else '?'
            _eval_lines.append(
                f'({_lbl}):\\quad '
                + ',\;'.join(_parts)
                + f'\;\;\\Sigma = {_sum_str}'
                + f'\;\;\\text{{pipeline}} = {_pv_str}')

        # Numerical check: only emit if SHOW_NUMERIC is True.
        # The computation above still runs so hand_eval_results is
        # populated for downstream cells (8.1b', 9-quick).
        if SHOW_NUMERIC:
            display(Math(r'\;\;'.join(
                r'\text{' + l + '}' if l.startswith(r'\textbf') else l
                for l in _eval_lines)))
            print()  # visual separator
elif not SKIP_CELL:
    print('k < 2: no decomposition to display.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1b′  Term-by-term numerical comparison for a SINGLE diagram
# ═══════════════════════════════════════════════════════════════
# Select a diagram index below.  For each 2^|E_δ| term shown in
# cell 8.1b Step 5, this cell builds a callable and evaluates it
# at test points on two 1D slices.  The sum of all terms is then
# compared against the pipeline's per-diagram contribution from
# cell 8.1.  Any discrepancy reveals a mismatch in the δ-
# integration logic.
#
# This runs fast (one diagram, no τ-grid sweep), so iterate here
# before committing to the expensive 8.1c grid evaluation.

DIAG_IDX = 0  # ← choose diagram index ∈ {0, 1, ..., len(all_unique)-1}

from msrjd.integration.time_domain import (
    build_G_t_matrix, G_t_delta_coeff, G_t_entry,
)
from msrjd.integration.time_domain.final_integral import (
    _lookup_prop_indices, _loop_number_from_graph,
)
from sage.all import fast_callable, CDF, SR, latex, solve, heaviside
from IPython.display import display, Math
import numpy as np
try:
    from scipy.integrate import quad, nquad
except ImportError:
    quad = nquad = None



# ── Early exit if hand_eval_results is empty ──
if not hand_eval_results:
    print("Cell 8.1b' SKIPPED — hand_eval_results is empty.")
    print("Set SKIP_CELL=False in cell 8.1b to populate it,")
    print("or just skip this cell (it is purely diagnostic).")
else:

    assert DIAG_IDX < len(all_unique), (
        f'DIAG_IDX={DIAG_IDX} but only {len(all_unique)} diagrams exist.')
    td = all_unique[DIAG_IDX]
    assert _loop_number_from_graph(td) == 0, (
        f'Diagram {DIAG_IDX} is loop-level, not tree. Pick a tree diagram.')

    # ── Setup (mirrors 8.1b main loop) ──
    _t_sym = SR.var('_t_cmp_')
    _G_obj = build_G_t_matrix(propagator_data, _t_sym, num_params=num_params)
    _pop_list = list(HAWKES_MODEL['index_sets']['pop'])

    D = td.prediagram[0]
    leaves = list(td.prediagram[2])
    leaf_set = set(leaves)
    internal = [v for v in D.vertices() if v not in leaf_set]

    ext_syms = [SR.var(f't_{j+1}', latex_name=f't_{{{j+1}}}')
                for j in range(k)]
    vertex_time = {}
    _used_canon = set()
    _leaf_to_canon = {}
    for j, lf in enumerate(leaves):
        field = td.external_legs.get(lf)
        for cp in range(len(external_fields)):
            if cp not in _used_canon and external_fields[cp] == field:
                _leaf_to_canon[j] = cp
                _used_canon.add(cp)
                break
        else:
            _leaf_to_canon[j] = j
    for j, lf in enumerate(leaves):
        vertex_time[lf] = ext_syms[_leaf_to_canon[j]]
    int_vars = []
    for v in internal:
        s_v = SR.var(f's_{v}', latex_name=f's_{{{v}}}')
        vertex_time[v] = s_v
        int_vars.append(s_v)

    edges = _gather_edges(td, _G_obj, vertex_time)
    pf_sr = diagram_prefactors[DIAG_IDX]
    pf_num = float(SR(pf_sr).subs(num_params))

    # Stationary substitutions (same as 8.1b Step 5)
    all_subs, tau_syms, u_syms, _ = build_step5_subs(ext_syms, int_vars, origin_idx=0)
    t_origin = ext_syms[0]

    print(f'Diagram {DIAG_IDX}:  prefactor = {pf_sr} = {pf_num:.6f}')
    print(f'  {len(edges)} edges, {sum(1 for e in edges if e["has_delta"])} with δ')
    print(f'  {len(int_vars)} internal variable(s): {[str(v) for v in int_vars]}')
    print(f'  τ symbols: {[str(t) for t in tau_syms]}')
    print(f'  u symbols: {[str(u) for u in u_syms]}')
    print()


    # ═══════════════════════════════════════════════════════════════
    # Look up precomputed results from cell 8.1b
    # ═══════════════════════════════════════════════════════════════
    if DIAG_IDX not in hand_eval_results:
        print(f'ERROR: Diagram {DIAG_IDX} not in hand_eval_results.')
        print(f'  Available: {sorted(hand_eval_results.keys())}')
        print(f'  Re-run cell 8.1b first.')
    else:
        pts_data = hand_eval_results[DIAG_IDX]
        test_points = sorted(pts_data.keys())
        test_labels = [f'\u03c4\u2081={pt[0]:g}, \u03c4\u2082={pt[1]:g}'
                       if len(pt) == 2 else f'\u03c4={pt[0]:g}'
                       for pt in test_points]

        header = ' ' * 8
        for lbl in test_labels:
            header += f'  {lbl:>18s}'
        print(header)
        print('\u2500' * len(header))

        n_terms = max(len(d['terms']) for d in pts_data.values())
        term_totals = [0.0] * len(test_points)

        for ti in range(n_terms):
            row = f'  T_{ti:<4d}'
            for pi, pt in enumerate(test_points):
                terms = pts_data[pt]['terms']
                if ti < len(terms):
                    t = terms[ti]
                    if t['val'] is not None:
                        row += f'  {t["val"]:>18.8e}'
                        term_totals[pi] += t['val']
                    else:
                        _delta_str = '\u03b4 (distributional)'
                        row += f'  {_delta_str:>18s}'
                else:
                    row += ' ' * 20
            if ti < len(pts_data[test_points[0]]['terms']):
                kind = pts_data[test_points[0]]['terms'][ti]['kind']
                row += f'   ({kind})'
            print(row)

        print('\u2500' * len(header))
        _sigma_lbl = '\u03a3 hand'
        row_sum = f'  {_sigma_lbl:>6s}'
        for val in term_totals:
            row_sum += f'  {val:>18.8e}'
        print(row_sum)

        _pipe_lbl = 'pipeline'
        row_pipe = f'  {_pipe_lbl:>8s}'
        for pt in test_points:
            pv = pts_data[pt]['pipeline']
            row_pipe += f'  {pv:>18.8e}' if pv is not None else f'  {"?":>18s}'
        print(row_pipe)

        row_diff = f'  {"diff":>8s}'
        for pi, pt in enumerate(test_points):
            pv = pts_data[pt]['pipeline']
            if pv is not None:
                row_diff += f'  {term_totals[pi] - pv:>18.8e}'
            else:
                row_diff += '                   \u2014'
        print(row_diff)

        row_ratio = f'  {"ratio":>8s}'
        for pi, pt in enumerate(test_points):
            pv = pts_data[pt]['pipeline']
            if pv is not None and abs(pv) > 1e-30:
                row_ratio += f'  {term_totals[pi] / pv:>18.8f}'
            else:
                row_ratio += '                   \u2014'
        print(row_ratio)
        print()

    # ── Also show pipeline total for ALL diagrams ──
    if '_td_result' in dir() and 'total_C' in _td_result:
        print('Pipeline total_C (all diagrams):')
        row_all = f'  {"total_C":>8s}'
        for pt in test_points:
            total_val = float(complex(_td_result['total_C'](0.0, *pt)).real)
            row_all += f'  {total_val:>18.8e}'
        print(row_all)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1c  Numerical evaluation of Phase J on the τ grid
# ═══════════════════════════════════════════════════════════════
# This cell evaluates the Phase J callable (built in cell 8.1) at
# each point on the τ grid.  THIS is the slow step — each point
# invokes scipy.integrate.quad on the stripped integrand.
#
# The symbolic decomposition (cell 8.1b) should be inspected first
# to verify the integrand structure looks correct.

# Gate: only runs in slice mode.  Theory curves on τ grid.
if SIM_MODE == 'slice':
    if k == 1:
        # k=1: the "slice" is a single scalar per external field --
        # <delta X_i> at stationarity.  Evaluate tree-only and full
        # (tree + 1-loop) totals separately so cell 32 can plot them
        # side by side along with the sim measurement.
        tau_phase_j = None
        C_tree_phase_j = None
        C_tree_phase_j_slices = None
        # callable signatures are (t_1,); value is stationary so the
        # argument is irrelevant (origin pinned t_1 = 0 anyway).
        _fn_full = _td_result['total_C']
        _fn_tree = _td_tree_result['total_C']
        C_theory_k1_total = float(complex(_fn_full(0.0)).real)
        C_theory_k1_tree  = float(complex(_fn_tree(0.0)).real)
        C_theory_k1_loop  = C_theory_k1_total - C_theory_k1_tree
        _field_label = f'{external_fields[0][0]}_{external_fields[0][1]}'
        print(f'Phase J theory (k=1, <{_field_label}>):')
        print(f'  tree    : {C_theory_k1_tree:+.6e}')
        print(f'  1-loop  : {C_theory_k1_loop:+.6e}')
        print(f'  total   : {C_theory_k1_total:+.6e}')
    
    elif k == 2:
        _total_C_fn = _td_result['total_C']
        tau_phase_j = np.array(tau_residue, dtype=float)
        C_tree_phase_j = np.zeros_like(tau_phase_j, dtype=float)
        # Pipeline convention: position i of total_C = time of external_fields[i].
        # With origin_leaf_idx=0, position 0 is pinned to t=0, position 1 is
        # the free τ. So for k=2 we evaluate total_C(0, τ), not total_C(τ, 0).
        for _i, _tv in enumerate(tau_phase_j):
            _val = _total_C_fn(0.0, float(_tv))
            C_tree_phase_j[_i] = float(np.real(_val))
        # Surviving deltas excluded from smooth evaluation (Θ(0) = 0 convention).
        if _td_result['delta_contributions']:
            print(f'  Note: {len(_td_result["delta_contributions"])} surviving '
                  f'delta(s) present but excluded from smooth grid.')
        C_tree_phase_j_slices = None
        _mid_j = len(tau_phase_j) // 2
        print(f"Phase J tree  C(0) = {C_tree_phase_j[_mid_j]:.6e}")
        if 'C_tree_residue' in globals() and C_tree_residue is not None and len(C_tree_residue) == len(tau_phase_j):
            _abs_err = float(np.max(np.abs(C_tree_phase_j - C_tree_residue)))
            print(f"max |Phase J - Phase I residue| = {_abs_err:.3e}")
    
    elif k >= 3:
        _total_C_fn = _td_result['total_C']
        tau_phase_j = np.array(tau_residue, dtype=float)
        C_tree_phase_j_slices = {}
    
        # ── Per-group contribution at test points ──
        print()
        print('Per-group smooth contributions:')
        # k-agnostic test points: origin (0) + (k-1) values of ±2
        _n_free = k - 1
        _test_pts = [
            tuple([0.0] + [2.0] * _n_free),
            tuple([0.0] + [2.0] * (_n_free - 1) + [-2.0]) if _n_free >= 1 else (0.0,),
            tuple([0.0] + [-2.0] + [2.0] * (_n_free - 1)) if _n_free >= 2 else (0.0, -2.0),
        ]
        # Deduplicate in case k is small
        _test_pts = list({p: None for p in _test_pts}.keys())
        for _tp in _test_pts:
            print(f'  At {_tp}:')
            _total_at_pt = 0.0
            for _gi, _g_info in enumerate(_td_result['groups']):
                if _g_info.get('contribution') is not None:
                    _val = _g_info['contribution'](*[float(x) for x in _tp])
                    _total_at_pt += float(complex(_val).real)
                    print(f'    Group {_gi}: {float(complex(_val).real):.6e}')
                else:
                    print(f'    Group {_gi}: SKIPPED ({_g_info.get("reason","?")})')
            print(f'    TOTAL:    {_total_at_pt:.6e}')
            _total_C_val = _td_result['total_C'](*[float(x) for x in _tp])
            print(f'    total_C:  {float(complex(_total_C_val).real):.6e}')
    
        # ── Parallel evaluation across τ points (parallel-eval branch) ─
        # Uses the fork-ProcessPool backend on ``total_C_batch`` added
        # by the parallel-eval branch.  Bit-identical to the serial
        # path (see test_phase_J_total_C_batch_parallel_matches_serial)
        # and typically 3-8x faster on a multi-core machine.
        #
        # If running against a main/checkout WITHOUT total_C_batch
        # (e.g. after a rollback), this falls back transparently to a
        # plain Python loop over total_C.
        import time as _time
        import os

        _n_workers = min(os.cpu_count() or 4, len(tau_phase_j))
        _tau_arr = np.array(tau_phase_j, dtype=float)
        _n_tau = len(_tau_arr)

        # Build list of slice specs (same semantics as before).
        import itertools as _itertools_sl
        _n_ext = k - 1
        _n_fixed = _n_ext - 1
        _slice_specs = []
        if _n_fixed <= 0:
            for _s in range(_n_ext):
                _slice_specs.append((_s, ()))
        else:
            for _fixed_tuple in _itertools_sl.product(TAU_FIXED_LIST, repeat=_n_fixed):
                for _s in range(_n_ext):
                    _slice_specs.append((_s, tuple(float(x) for x in _fixed_tuple)))

        _batch_fn = _td_result.get('total_C_batch')
        _parallel_mode = (TD_PARALLEL and _batch_fn is not None)
        _status_workers = (TD_N_WORKERS if TD_N_WORKERS is not None
                           else min(os.cpu_count() or 4, len(tau_phase_j)))
        if _parallel_mode:
            print(f'  Phase J evaluation: parallel=True, '
                  f'workers={_status_workers}')
        else:
            print(f'  Phase J evaluation: parallel=False (serial)')
        _t0_slices = _time.perf_counter()

        for _s, _fixed_tuple in _slice_specs:
            # Build τ-tuples for this slice: position 0 pinned to 0.0,
            # the sweep axis (_s) takes the varying value, the others
            # take the fixed_tuple entries in order.
            _tau_tuples = []
            for _tv in _tau_arr:
                _time_args = [0.0]
                _fixed_idx = 0
                for _j in range(_n_ext):
                    if _j == _s:
                        _time_args.append(float(_tv))
                    else:
                        _time_args.append(float(_fixed_tuple[_fixed_idx]))
                        _fixed_idx += 1
                _tau_tuples.append(tuple(_time_args))

            if _batch_fn is not None:
                # Parallel fast path (parallel-eval branch) — the
                # two knobs come from the Configuration cell.
                _vals = _batch_fn(_tau_tuples,
                                  parallel=TD_PARALLEL,
                                  n_workers=TD_N_WORKERS)
            else:
                # Fallback: serial loop over total_C for main branches
                # that don't yet have the batch API.
                _vals = [_td_result['total_C'](*_pt) for _pt in _tau_tuples]
            _C_slice = np.array([float(complex(v).real) for v in _vals])

            C_tree_phase_j_slices[(_s, _fixed_tuple)] = _C_slice
            _elapsed = _time.perf_counter() - _t0_slices
            _fixed_str = (','.join(f'{x:+g}' for x in _fixed_tuple)
                          if _fixed_tuple else '—')
            print(f'  Slice (s={_s}, fixed=({_fixed_str})): done '
                  f'({_elapsed:.1f}s elapsed)')

        _total_elapsed = _time.perf_counter() - _t0_slices
        _total_elapsed = _time.perf_counter() - _t0_slices
        print(f'  All slices: {_total_elapsed:.1f}s total '
              f'({_n_workers} workers, {_n_tau} grid points)')
    
        # Surviving deltas excluded from smooth evaluation (Θ(0) = 0 convention).
        if _td_result['delta_contributions']:
            print(f'  Note: {len(_td_result["delta_contributions"])} surviving '
                  f'delta(s) present but excluded from smooth slices.')
    
        # Pick a representative slice for C_tree_phase_j (the first one)
        _first_key = next(iter(C_tree_phase_j_slices))
        C_tree_phase_j = C_tree_phase_j_slices[_first_key]
        _mid_j = len(tau_phase_j) // 2
        print()
        for _key, _curve in C_tree_phase_j_slices.items():
            _s_k, _fixed_k = _key
            _fstr = ','.join(f'{x:+g}' for x in _fixed_k) if _fixed_k else '—'
            print(f"Phase J tree  slice (s={_s_k}, fixed=({_fstr}))  "
                  f"at τ=0 = {_curve[_mid_j]:.6e}")
    
    else:
        tau_phase_j = None
        C_tree_phase_j = None
        C_tree_phase_j_slices = None
    

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.2  Theory vs Simulation — 4-slice comparison
# ═══════════════════════════════════════════════════════════════
# For k=3 with TAU_FIXED_LIST = [1.0, -1.0], this produces 4 panels:
#   (s=0, τ_fixed=+1): vary τ₁, fix τ₂ = +1
#   (s=0, τ_fixed=-1): vary τ₁, fix τ₂ = -1
#   (s=1, τ_fixed=+1): vary τ₂, fix τ₁ = +1
#   (s=1, τ_fixed=-1): vary τ₂, fix τ₁ = -1
# Gate: only runs in slice mode.  Plots theory vs sim slices.
if SIM_MODE == 'slice':
    import matplotlib.pyplot as plt
    
    if k >= 3 and C_tree_phase_j_slices is not None:
        n_ext = k - 1
        # Collect unique fixed tuples from the slices dict (new format)
        # or fall back to TAU_FIXED_LIST (k=3 with scalar fixed values).
        _fixed_tuples = sorted({key[1] for key in C_tree_phase_j_slices.keys()
                                if isinstance(key[1], tuple)}) or \
                        [(float(v),) for v in sorted(TAU_FIXED_LIST)]
        n_rows = len(_fixed_tuples)
        _tau_max_plot = float(tau_max) if 'tau_max' in dir() else 50.0
        _corr_label = corr_label if 'corr_label' in dir() else ', '.join(f'{f[0]}_{f[1]}' for f in external_fields)
    
        fig, axes = plt.subplots(n_rows, n_ext,
                                 figsize=(7 * n_ext, 5 * n_rows),
                                 squeeze=False)
    
        for row, fixed_tuple in enumerate(_fixed_tuples):
            for s in range(n_ext):
                ax = axes[row, s]
    
                # Simulation (available after cell 9 runs)
                try:
                    sim_key = (s, fixed_tuple)
                    sim_curve = C_sim_slices.get(sim_key)
                    if sim_curve is not None:
                        ax.plot(tau_sim_grid, sim_curve, 'k-', lw=1.5, alpha=0.7,
                                label=f'Simulation ({N_RUNS} runs)')
                except NameError:
                    pass  # simulation hasn't run yet
    
                # Theory (Phase J pipeline)
                pj_key = (s, fixed_tuple)
                if pj_key in C_tree_phase_j_slices and tau_phase_j is not None:
                    mask_j = np.abs(tau_phase_j) <= _tau_max_plot
                    ax.plot(tau_phase_j[mask_j],
                            C_tree_phase_j_slices[pj_key][mask_j],
                            'm-', lw=2, label='Theory (tree-level)')
    
                # Labels
                # Generic: vary τ_{s+1}; fix all other τ's at values from
                # fixed_tuple (paired in order with the not-swept axes).
                sweep_idx = s + 1
                other_idxs = [j + 1 for j in range(n_ext) if j != s]
                other_str = ', '.join(
                    rf'$\tau_{{{j}}}={v:+g}$'
                    for j, v in zip(other_idxs, fixed_tuple)
                )
                title = rf'vary $\tau_{{{sweep_idx}}}$, ' + other_str
                xlabel = rf'$\tau_{{{sweep_idx}}}$'
    
                ax.set_xlabel(xlabel, fontsize=13)
                ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
                ax.set_title(title, fontsize=12)
                ax.legend(fontsize=9)
                ax.grid(True, alpha=0.3)
                ax.axhline(0, color='k', lw=0.5)
    
        fig.suptitle(
            rf'$k={k}$ tree-level cumulant: $\langle {_corr_label} \rangle$',
            fontsize=14)
        plt.tight_layout()
        plt.show()
    
    elif k == 2 and C_tree_phase_j is not None:
        _tau_max_plot = float(tau_max) if 'tau_max' in dir() else 50.0
        _corr_label = corr_label if 'corr_label' in dir() else ', '.join(f'{f[0]}_{f[1]}' for f in external_fields)
        fig, ax = plt.subplots(1, 1, figsize=(8, 5))
        _sim = C_sim_slices.get(0) if 'C_sim_slices' in dir() and C_sim_slices else None
        if _sim is not None:
            ax.plot(tau_sim_grid, _sim, 'k-', lw=1.5, alpha=0.7,
                    label=f'Simulation ({N_RUNS} runs)')
        if tau_phase_j is not None:
            mask_j = np.abs(tau_phase_j) <= _tau_max_plot
            ax.plot(tau_phase_j[mask_j], C_tree_phase_j[mask_j],
                    'm-', lw=2, label='Theory (tree-level)')
        ax.set_xlabel(r'$\tau$', fontsize=13)
        ax.set_ylabel(r'$C^{(2)}(\tau)$', fontsize=13)
        ax.set_title(f'$k=2$ correlator: {_corr_label}', fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', lw=0.5)
        plt.tight_layout()
        plt.show()
    else:
        print(f'Slice comparison plot skipped (k={k}).')
    

### 9. Simulation comparison

Simulate the 2-population nonlinear Hawkes process with the same
fundamental parameters and compare:
- Mean firing rates: simulation vs MF + 1-loop
- Cross-covariance $C_{12}(\tau)$: simulation vs tree + 1-loop

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9-quick  Point-evaluation: theory vs short simulation
# ═══════════════════════════════════════════════════════════════
# Instead of computing full τ-grid slices, evaluate theory and a
# short simulation at specific test points.  Much faster for
# debugging the δ-integration pipeline.

if SIM_MODE == 'point':
    import numpy as np
    from numpy.random import default_rng
    import time as _time

    # ── Test points: use TEST_POINTS from cell 2 if set, else k-specific defaults ──
    try:
        TEST_POINTS  # set by cell 2 config
    except NameError:
        TEST_POINTS = None
    if TEST_POINTS is None:
        if k == 4:
            TEST_POINTS = [(4.0, 2.0, -2.0)]
        elif k == 3:
            TEST_POINTS = [(-20.0, 1.0), (-20.0, -1.0), (-20.0, 5.0), (-20.0, -5.0)]
        elif k == 2:
            TEST_POINTS = [(5.0,), (10.0,), (20.0,), (50.0,)]
        else:
            TEST_POINTS = [tuple([1.0] * (k - 1))]
    # Auto-generate labels from the points themselves.
    def _fmt_pt(pt):
        return ','.join(f'τ{i+1}={v:+g}' for i, v in enumerate(pt))
    TEST_LABELS = [_fmt_pt(pt) for pt in TEST_POINTS]

    # ── Model parameters ──
    E_sim     = [float(x) for x in fundamental['E']]
    w_sim     = [[float(x) for x in row] for row in fundamental['w']]
    tau_sim   = float(fundamental['tau'])
    tau_g_sim = float(fundamental['tau_g'])
    a_sim     = float(fundamental['a'])
    npop_sim  = len(E_sim)

    # ── Simulation settings: inherit from cell 2 config; local fallbacks if unset ──
    try:
        N_RUNS
    except NameError:
        N_RUNS = 3
    try:
        T_sim
    except NameError:
        T_sim = float(5_000_000)
    try:
        dt_sim
    except NameError:
        dt_sim = float(0.01)
    try:
        dt_bin
    except NameError:
        dt_bin = float(1.0)
    # Seed is ALWAYS randomized per run (no fixed-seed option).
    import secrets as _secrets
    BASE_SEED = _secrets.randbits(31)

    from models.hawkes_sim_quad_expg_numba import sim_hawkes_quad_expg_numba
    from models.cumulant_estimator import compute_kpoint_slice

    n_steps = int(T_sim / dt_sim)
    bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
    dt_bin_eff = bin_size_steps * dt_sim
    n_bins = n_steps // bin_size_steps
    v_init = np.array([float(vstar_vals[i]) for i in range(npop_sim)])
    E_arr = np.array(E_sim, dtype=float)
    W_arr = np.array(w_sim, dtype=float)
    pop_indices = [ef[1] - 1 for ef in external_fields]
    field_types = [ef[0] for ef in external_fields]

    # JIT warmup
    _ = sim_hawkes_quad_expg_numba(int(1000), float(dt_sim), float(tau_sim),
                              float(tau_g_sim), float(a_sim),
                              E_arr, W_arr,
                              v_init.copy(), int(bin_size_steps), int(100), int(0))

    print(f'Quick simulation: {N_RUNS} runs × T={T_sim:.0f}')
    print(f'  dt_sim={dt_sim}, dt_bin_eff={dt_bin_eff}, n_bins={n_bins:,}')
    print()

    # ── Run simulations and extract cumulant at test points ──
    # For each test point (τ₁, τ₂), we use compute_kpoint_slice with
    # the sweep axis at one of the τ values and the other fixed.
    # Strategy: for each point, slice 0 (sweep τ₁, fix τ₂) and
    # interpolate at the desired τ₁.

    max_lag_bins = int(max(abs(p) for pt in TEST_POINTS for p in pt) / dt_bin_eff) + 5

    sim_values = {lbl: [] for lbl in TEST_LABELS}

    _t0 = _time.perf_counter()
    for run_idx in range(N_RUNS):
        seed = BASE_SEED + run_idx * 1000
        binned_counts_run, voltage_bins_run, total_spikes_run = sim_hawkes_quad_expg_numba(
            int(n_steps), float(dt_sim), float(tau_sim),
            float(tau_g_sim), float(a_sim),
            E_arr, W_arr,
            v_init.copy(), int(bin_size_steps), int(n_bins), int(seed),
        )

        for pi_idx, (pt, lbl) in enumerate(zip(TEST_POINTS, TEST_LABELS)):
            # Generic: pt is a (k-1)-tuple of τ values.
            # Sweep the FIRST τ (sweep_s=0); fix the remaining (k-2) τ's.
            tau_sweep = float(pt[0])
            fixed_taus = [float(x) for x in pt[1:]]
            fixed_bins = [int(round(x / dt_bin_eff)) for x in fixed_taus]
            # Build lag_bins: position 0 = origin (0), position 1 = sweep (None),
            # positions 2..k-1 = fixed bins in order.
            lag_bins = [0, None] + fixed_bins
            try:
                tau_grid, c_slice = compute_kpoint_slice(
                    binned_counts_run, float(dt_bin_eff),
                    [int(p) for p in pop_indices],
                    lag_bins, int(max_lag_bins),
                    field_types=field_types,
                    voltage_bins=voltage_bins_run,
                )
                # Interpolate at the sweep value
                val = float(np.interp(tau_sweep, tau_grid, c_slice))
                sim_values[lbl].append(val)
            except Exception as e:
                print(f'  Run {run_idx} point {lbl}: {e}')
                sim_values[lbl].append(np.nan)

    _elapsed = _time.perf_counter() - _t0
    print(f'Simulation done in {_elapsed:.1f}s')
    print()

    # ── Theory values (from Phase J pipeline) ──
    _total_C_fn = _td_result['total_C']

    # ── Results table ──
    print(f'{"Point":>18s}  {"Theory":>14s}  {"Sim (mean)":>14s}  {"Sim (std)":>14s}  {"Ratio":>10s}')
    print('─' * 78)
    for pt, lbl in zip(TEST_POINTS, TEST_LABELS):
        theory_val = float(complex(_total_C_fn(0.0, *pt)).real)
        sim_vals = sim_values[lbl]
        sim_mean = float(np.nanmean(sim_vals))
        sim_std = float(np.nanstd(sim_vals))
        ratio = theory_val / sim_mean if abs(sim_mean) > 1e-30 else float('nan')
        print(f'{lbl:>18s}  {theory_val:>14.6e}  {sim_mean:>14.6e}  {sim_std:>14.6e}  {ratio:>10.4f}')

    # ── Per-diagram theory breakdown ──
    print()
    print('Per-diagram theory (pipeline) contributions:')
    header = f'{"Diagram":>8s}'
    for lbl in TEST_LABELS:
        header += f'  {lbl:>16s}'
    print(header)
    print('─' * len(header))

    for gi, g_info in enumerate(_td_result['groups']):
        if g_info.get('contribution') is None:
            continue
        row = f'  D_{gi:<4d}'
        for pt in TEST_POINTS:
            val = float(complex(g_info['contribution'](0.0, *pt)).real)
            row += f'  {val:>16.6e}'
        print(row)

    row_total = f'  {"TOTAL":>6s}'
    for pt in TEST_POINTS:
        val = float(complex(_total_C_fn(0.0, *pt)).real)
        row_total += f'  {val:>16.6e}'
    print(row_total)

    # ── Hand term sums (from 8.1b precomputed results) ──
    if 'hand_eval_results' in dir() and hand_eval_results:
        print()
        print('Per-diagram hand term sums (from 8.1b):')
        print(header)
        print('─' * len(header))

        hand_grand_total = [0.0] * len(TEST_POINTS)
        for gi in sorted(hand_eval_results.keys()):
            row = f'  D_{gi:<4d}'
            for pi_idx, pt in enumerate(TEST_POINTS):
                pt_key = tuple(float(x) for x in pt)
                if pt_key in hand_eval_results[gi]:
                    val = hand_eval_results[gi][pt_key]['total']
                    row += f'  {val:>16.6e}'
                    hand_grand_total[pi_idx] += val
                else:
                    row += f'  {"N/A":>16s}'
            print(row)

        row_ht = f'  {"TOTAL":>6s}'
        for val in hand_grand_total:
            row_ht += f'  {val:>16.6e}'
        print(row_ht)
    else:
        print('\nHand term sums not available (run cell 8.1b first).')

else:
    print(f"(point-mode cell skipped: SIM_MODE={SIM_MODE})")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9.  Simulation of the 2-population linear Hawkes process
# ═══════════════════════════════════════════════════════════════
# Runs N_RUNS independent simulations with different RNG seeds and
# averages the k-point cumulant slices to reduce noise.
# Gate: only runs in slice mode.  Long simulation for slice plots.
if SIM_MODE == 'slice':
    import numpy as np
    from numpy.random import default_rng
    import time as _time
    
    # ── Simulation parameters: inherit from cell 2 config; per-k fallbacks ──
    try:
        N_RUNS
    except NameError:
        N_RUNS = 5
    try:
        T_sim
    except NameError:
        if k <= 2:
            T_sim = float(200_000)
        else:
            T_sim = float(10_000_000)
    try:
        dt_sim
    except NameError:
        dt_sim = float(0.01) if k > 2 else float(0.02)
    try:
        dt_bin
    except NameError:
        dt_bin = float(1.0) if k > 2 else float(0.25)
    try:
        tau_max
    except NameError:
        tau_max = float(50)
    # Seed is ALWAYS randomized per run (no fixed-seed option).
    import secrets as _secrets_32
    BASE_SEED = _secrets_32.randbits(31)
    
    # ── Model parameters ──
    E_sim     = [float(x) for x in fundamental['E']]
    w_sim     = [[float(x) for x in row] for row in fundamental['w']]
    tau_sim   = float(fundamental['tau'])
    tau_g_sim = float(fundamental['tau_g'])
    a_sim     = float(fundamental['a'])
    npop_sim  = len(E_sim)
    
    def phi_sim(v_val):
        """Transfer function: phi(v) = a * v^2 (quadratic with gain)."""
        return a_sim * v_val * v_val
    
    from models.hawkes_sim_quad_expg_numba import sim_hawkes_quad_expg_numba
    
    # ── Derived quantities ──
    n_steps = int(T_sim / dt_sim)
    bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
    dt_bin_eff = bin_size_steps * dt_sim
    n_bins = n_steps // bin_size_steps
    v_init = np.array([float(vstar_vals[i]) for i in range(npop_sim)])
    E_arr = np.array(E_sim, dtype=float)
    W_arr = np.array(w_sim, dtype=float)
    
    print(f'Simulation: {N_RUNS} runs × T={T_sim:.0f} (total {N_RUNS*T_sim:.0f})')
    print(f'  dt_sim={dt_sim}, dt_bin_eff={dt_bin_eff}, n_steps={n_steps:,}, n_bins={n_bins:,}')
    
    # JIT warmup
    _ = sim_hawkes_quad_expg_numba(int(1000), float(dt_sim), float(tau_sim),
                              float(tau_g_sim), float(a_sim),
                              E_arr, W_arr,
                              v_init.copy(), int(bin_size_steps), int(100), int(0))
    
    # ── Population indices and labels ──
    max_lag_bins = int(tau_max / dt_bin_eff)
    tau_sim_grid = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin_eff
    n_ext = k - 1
    pop_indices = [ef[1] - 1 for ef in external_fields]
    field_types = [ef[0] for ef in external_fields]
    field_labels = [f'{ef[0]}_{ef[1]}' for ef in external_fields]
    corr_label = ', '.join(field_labels)
    
    if 'TAU_FIXED_LIST' not in globals():
        TAU_FIXED_LIST = [0.0] if k == 2 else [1.0, -1.0]
    
    # ── Multi-run loop ──
    from models.cumulant_estimator import compute_kpoint_slice
    
    all_run_rates = []
    all_run_slices = []  # list of dicts, one per run
    
    _t0_total = _time.perf_counter()
    for run_idx in range(N_RUNS):
        seed = BASE_SEED + run_idx * 1000
        _t0 = _time.perf_counter()
        binned_counts_run, voltage_bins_run, total_spikes_run = sim_hawkes_quad_expg_numba(
            int(n_steps), float(dt_sim), float(tau_sim),
            float(tau_g_sim), float(a_sim),
            E_arr, W_arr,
            v_init.copy(), int(bin_size_steps), int(n_bins), int(seed),
        )
        _elapsed = _time.perf_counter() - _t0
    
        T_used = n_bins * dt_bin_eff
        run_rates = total_spikes_run / T_used
        all_run_rates.append(run_rates)
        print(f'  Run {run_idx+1}/{N_RUNS} (seed={seed}): {_elapsed:.1f}s, '
              f'rates=[{", ".join(f"{r:.4f}" for r in run_rates)}]')
    
        # Compute cumulant slices for this run
        run_slices = {}
        if k >= 2:
            import itertools as _itertools_sim
            _n_fixed = n_ext - 1  # k - 2
            if _n_fixed <= 0:
                # k == 2: single slice, no fixed values
                fixed_tuples_iter = [()]
            else:
                fixed_tuples_iter = list(_itertools_sim.product(
                    TAU_FIXED_LIST, repeat=_n_fixed))
    
            for fixed_tuple in fixed_tuples_iter:
                fixed_bins = tuple(int(round(x / dt_bin_eff)) for x in fixed_tuple)
                for s in range(n_ext):
                    # Build lag_bins generically: position 0 is origin,
                    # position s+1 is the sweep (None), other positions
                    # get fixed lag values in order.
                    lag_bins = [0]
                    fidx = 0
                    for j in range(n_ext):
                        if j == s:
                            lag_bins.append(None)
                        else:
                            lag_bins.append(fixed_bins[fidx])
                            fidx += 1
    
                    tau_grid_run, C_slice_run = compute_kpoint_slice(
                        binned_counts_run, float(dt_bin_eff),
                        [int(p) for p in pop_indices],
                        lag_bins, int(max_lag_bins),
                        field_types=field_types,
                        voltage_bins=voltage_bins_run,
                    )
                    key = s if k == 2 else (s, tuple(float(x) for x in fixed_tuple))
                    run_slices[key] = C_slice_run
        all_run_slices.append(run_slices)
    
    _elapsed_total = _time.perf_counter() - _t0_total
    print(f'All {N_RUNS} runs done in {_elapsed_total:.1f}s total.')
    
    # ── Average across runs ──
    sim_rates = np.mean(all_run_rates, axis=0)
    # Keep the last run's binned_counts for any downstream use
    binned_counts = binned_counts_run
    
    C_sim_slices = {}
    if all_run_slices:
        all_keys = all_run_slices[0].keys()
        for key in all_keys:
            stacked = np.array([run[key] for run in all_run_slices])
            C_sim_slices[key] = np.nanmean(stacked, axis=0)
    
    # ── Mask sim data at surviving-delta tau values ──
    if k >= 3 and '_td_result' in dir() and _td_result.get('delta_contributions'):
        for key in list(C_sim_slices.keys()):
            if isinstance(key, tuple) and len(key) == 2:
                s, fixed_tuple = key
                # Build the fixed_values dict indexed by free-ext position
                # (free-ext = positions 1..k-1 of the time args; index 0..k-2)
                if not isinstance(fixed_tuple, tuple):
                    fixed_tuple = (fixed_tuple,)  # backward compat
                _vary_free = s  # index of sweep axis in free-ext space
                _fixed_d_sim = {}
                fidx = 0
                for j in range(n_ext):
                    if j == s:
                        continue
                    _fixed_d_sim[j] = float(fixed_tuple[fidx]) if fidx < len(fixed_tuple) else 0.0
                    fidx += 1
                _forbidden_sim = _forbidden_tau_values(
                    _td_result['delta_contributions'], _vary_free, _fixed_d_sim)
                _dtau_sim = abs(tau_sim_grid[1] - tau_sim_grid[0]) if len(tau_sim_grid) > 1 else 1.0
                C_slice = C_sim_slices[key]
                for _ftv in _forbidden_sim:
                    if _ftv == 'ALL':
                        C_slice[:] = np.nan
                    else:
                        _mask_radius = max(2.0 * _dtau_sim, float(dt_bin_eff))
                        _near_mask = np.abs(tau_sim_grid - float(_ftv)) <= _mask_radius
                        C_slice[_near_mask] = np.nan
                C_sim_slices[key] = C_slice
    
    print(f'\nAveraged rates: {[f"{r:.4f}" for r in sim_rates]}')
    print(f'MF rates:       {[f"{nstar_vals[i]:.4f}" for i in range(npop_sim)]}')
    print(f'Cumulant slices: {len(C_sim_slices)} (averaged over {N_RUNS} runs)')
    
    
    # ═══════════════════════════════════════════════════════════════
    # Comparison plots
    # ═══════════════════════════════════════════════════════════════
    import matplotlib.pyplot as plt
    
    mf_rates = [float(nstar_vals[i]) for i in range(npop_sim)]
    
    def _plot_mean_rates(ax):
        x_pos = np.arange(npop_sim)
        width = 0.25
        ax.bar(x_pos - width, sim_rates, width, label='Simulation',
               color='#4CAF50', edgecolor='k', linewidth=0.5)
        ax.bar(x_pos, mf_rates, width, label='Mean-field',
               color='#2196F3', edgecolor='k', linewidth=0.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
        ax.set_ylabel('Firing rate', fontsize=12)
        ax.set_title('Mean firing rates', fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
    
    
    if k == 1:
        # k=1 plot: mean firing rate (or mean voltage) of the chosen
        # external field.  Theory is shown as a STACKED bar:
        #   bottom = mean-field (tree) value:  n*_i or v*_i
        #   top    = 1-loop correction:        <delta X_i>_1loop
        # so the full bar height = MF + 1-loop = total theory mean.
        # The sim bar is the raw measured mean (not the deviation) with
        # an error bar from the inter-run std when multiple runs exist.
        _ef = external_fields[0]
        _pop = _ef[1] - 1
        if _ef[0] == 'dn':
            _mf_value   = float(nstar_vals[_pop])
            _sim_value  = float(sim_rates[_pop])
            _sim_std    = 0.0
            if 'all_run_rates' in dir() and len(all_run_rates) > 1:
                _arr = np.array([r[_pop] for r in all_run_rates])
                _sim_value = float(np.mean(_arr))
                _sim_std   = float(np.std(_arr, ddof=1))
            _field_sym = rf'n_{{{_ef[1]}}}'
        else:   # 'dv'
            _mf_value   = float(vstar_vals[_pop])
            _sim_value  = _mf_value  # fallback
            _sim_std    = 0.0
            if 'voltage_bins_run' in dir():
                _sim_value = float(voltage_bins_run[_pop].mean())
            _field_sym = rf'v_{{{_ef[1]}}}'

        # 1-loop correction (scalar from cell 28).  Tree-level
        # fluctuation is 0 at the MF saddle; we collapse it into the
        # MF bar for display simplicity.  If it's numerically non-zero
        # the sanity print in cell 28 will show it.
        _loop_corr  = float(C_theory_k1_loop)  if 'C_theory_k1_loop'  in dir() else 0.0
        _theory_tot = _mf_value + _loop_corr

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        _plot_mean_rates(axes[0])

        ax = axes[1]
        xs = np.array([0, 1])
        width = 0.55

        # Stacked theory bar: MF (blue) + 1-loop (purple, may be negative)
        ax.bar(xs[0], _mf_value, width=width, color='#2196F3',
               edgecolor='k', linewidth=0.7, label='Mean-field (tree)')
        ax.bar(xs[0], _loop_corr, width=width, bottom=_mf_value,
               color='#9C27B0', edgecolor='k', linewidth=0.7,
               label='1-loop correction')

        # Sim bar (green) with inter-run std error bar
        ax.bar(xs[1], _sim_value, width=width, yerr=_sim_std,
               color='#4CAF50', edgecolor='k', linewidth=0.7,
               capsize=6, label=f'Simulation ({N_RUNS} runs)')

        # Annotations: MF value inside MF bar, 1-loop value inside
        # 1-loop bar, total at the very top, sim value at top of sim.
        _span = max(abs(_theory_tot), abs(_sim_value), 1e-12)
        _lbl_offset = 0.02 * _span
        if abs(_mf_value) > 1e-12:
            ax.text(xs[0], _mf_value / 2,
                    f'MF\n{_mf_value:.4f}',
                    ha='center', va='center', fontsize=10,
                    color='white', fontweight='bold')
        if abs(_loop_corr) > 0.02 * _span:
            ax.text(xs[0], _mf_value + _loop_corr / 2,
                    f'1-loop\n{_loop_corr:+.3e}',
                    ha='center', va='center', fontsize=9,
                    color='white', fontweight='bold')
        else:
            # 1-loop too small to label inside -- annotate outside
            ax.text(xs[0], _theory_tot + _lbl_offset,
                    f'1-loop = {_loop_corr:+.3e}',
                    ha='center', va='bottom', fontsize=8, color='#9C27B0')
        ax.text(xs[0], _theory_tot + 2 * _lbl_offset,
                f'total = {_theory_tot:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.text(xs[1], _sim_value + 2 * _lbl_offset,
                f'{_sim_value:.4f}' + (f' ± {_sim_std:.1e}' if _sim_std > 0 else ''),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

        ax.set_xticks(xs)
        ax.set_xticklabels(['Theory\n(MF + 1-loop)',
                            f'Simulation'], fontsize=11)
        ax.set_ylabel(rf'$\langle {_field_sym} \rangle$', fontsize=13)
        ax.set_title(rf'$k=1$ mean: $\langle {_field_sym} \rangle$',
                     fontsize=13)
        ax.grid(True, alpha=0.3, axis='y')
        ax.axhline(0, color='k', lw=0.6)
        ax.legend(loc='best', fontsize=10)

        plt.tight_layout()
        plt.show()

    elif n_ext == 1:
        # Original k=2 plot (n_ext = k - 1 = 1): the cross/auto
        # covariance as a function of a single tau axis.
        n_plots = 1
        fig, axes = plt.subplots(1, 1 + n_plots,
                                 figsize=(7 * (1 + n_plots), 5))
        if not isinstance(axes, np.ndarray):
            axes = [axes]
        _plot_mean_rates(axes[0])

        for s in range(n_ext):
            ax = axes[1 + s]
            _sim = C_sim_slices.get(s)
            if _sim is not None:
                ax.plot(tau_sim_grid, _sim, 'k-', lw=1.5, alpha=0.7,
                        label=f'Simulation ({N_RUNS} runs avg)')

            if tau_phase_j is not None and C_tree_phase_j is not None:
                _mask_j = np.abs(tau_phase_j) <= tau_max
                ax.plot(tau_phase_j[_mask_j], C_tree_phase_j[_mask_j],
                        'm-', lw=2, label='Theory (tree-level)')

            ax.set_xlabel(r'$\tau_1$', fontsize=13)
            is_auto = (len(external_fields) >= 2
                       and external_fields[0] == external_fields[1])
            ctype = 'Auto' if is_auto else 'Cross'
            ax.set_title(f'{ctype}-covariance: {corr_label}', fontsize=13)
            ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
            ax.axhline(0, color='k', lw=0.5)

        plt.tight_layout()
        plt.show()
    
    else:
        # Collect unique fixed-τ tuples from the slice keys (new format:
        # (s, fixed_tuple) where fixed_tuple is a length-(k-2) tuple of floats).
        _fixed_tuples = sorted({k_[1] for k_ in C_sim_slices.keys()
                                if isinstance(k_, tuple)
                                and isinstance(k_[1], tuple)}) \
            if C_sim_slices else [tuple([0.0] * max(1, n_ext - 1))]
        if not _fixed_tuples:
            _fixed_tuples = [tuple([0.0] * max(1, n_ext - 1))]
        _n_rows = len(_fixed_tuples)
        fig, axes = plt.subplots(_n_rows, 1 + n_ext,
                                 figsize=(7 * (1 + n_ext), 5 * _n_rows),
                                 squeeze=False)
        _plot_mean_rates(axes[0, 0])
        for _r in range(1, _n_rows):
            axes[_r, 0].axis('off')
    
        for _r, fixed_tuple in enumerate(_fixed_tuples):
            for s in range(n_ext):
                ax = axes[_r, 1 + s]
    
                # Key format matches what compute_kpoint_slice + cell 28
                # build: (s, fixed_tuple) with fixed_tuple a float-tuple.
                _sim_key = (s, tuple(float(x) for x in fixed_tuple))
                _sim = C_sim_slices.get(_sim_key)
                if _sim is not None:
                    ax.plot(tau_sim_grid, _sim, 'k-', lw=1.5, alpha=0.7,
                            label=f'Simulation ({N_RUNS} runs avg)')
    
                _pj_key = _sim_key
                if ('C_tree_phase_j_slices' in globals()
                        and C_tree_phase_j_slices is not None
                        and _pj_key in C_tree_phase_j_slices
                        and tau_phase_j is not None):
                    _mask_j = np.abs(tau_phase_j) <= tau_max
                    ax.plot(tau_phase_j[_mask_j],
                            C_tree_phase_j_slices[_pj_key][_mask_j],
                            'm-', lw=2, label='Theory (tree-level)')
    
                # Labels: vary τ_{s+1}; show fixed axes with their values
                # paired 1-for-1 with fixed_tuple entries (same ordering as
                # cell 28's evaluation loop).
                sweep_idx = s + 1
                other_idxs = [j + 1 for j in range(n_ext) if j != s]
                other_label = ', '.join(
                    rf'$\tau_{{{j}}}={v:+g}$'
                    for j, v in zip(other_idxs, fixed_tuple)
                )
                ax.set_xlabel(rf'$\tau_{{{sweep_idx}}}$', fontsize=13)
                ax.set_title(f'$k={k}$ slice: vary $\\tau_{{{sweep_idx}}}$, '
                             f'{other_label}', fontsize=12)
                ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
                ax.legend(fontsize=9)
                ax.grid(True, alpha=0.3)
                ax.axhline(0, color='k', lw=0.5)
    
        plt.tight_layout()
        plt.show()
    
    print(f'\n{"="*60}')
    print('Comparison summary')
    print(f'{"="*60}')
    print(f'  k = {k}, external_fields = {external_fields}')
    print(f'  {N_RUNS} runs × T={T_sim:.0f} = {N_RUNS*T_sim:.0f} total time')
    for i in range(npop_sim):
        mf = nstar_vals[i]
        sim_r = sim_rates[i]
        pct = 100 * (sim_r - mf) / mf if mf > 0 else 0
        print(f'  Pop {i+1}:  sim={sim_r:.4f}  MF={mf:.4f}  diff={pct:+.2f}%')
    

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.3  Heatmap comparison — DISABLED (finicky, revisit later)
# ═══════════════════════════════════════════════════════════════
if False:  # <-- set to True to re-enable heatmap comparison
    # Both theory and sim heatmaps are built row-by-row using the SAME
    # convention: fix τ₁ (= lag of field b from field a), sweep τ₂
    # (= lag of field c from field a). This ensures identical lag
    # conventions and normalization between theory and sim.
    
    import matplotlib.pyplot as plt
    
    if k == 3 and '_td_result' in dir():
        # ── Settings ──
        HEATMAP_TAU_MAX  = 30.0
        HEATMAP_TAU_STEP = 1.0
    
        tau_1d = np.arange(-HEATMAP_TAU_MAX, HEATMAP_TAU_MAX + HEATMAP_TAU_STEP/2,
                           HEATMAP_TAU_STEP)
        n_tau = len(tau_1d)
    
        # ── Theory heatmap: row-by-row, same convention as sim ──
        # For each τ₁ row, evaluate total_C(t_a=0, t_b=τ₁, t_c=τ₂) for
        # each τ₂ in tau_1d.
        _total_C_fn = _td_result['total_C']
        C_theory_2d = np.zeros((n_tau, n_tau))
        print(f'Evaluating theory heatmap ({n_tau}x{n_tau} = {n_tau**2} points)...')
        for i, t1 in enumerate(tau_1d):
            for j, t2 in enumerate(tau_1d):
                C_theory_2d[i, j] = float(np.real(
                    _total_C_fn(0.0, float(t1), float(t2))
                ))
        print(f'  Smooth only. Range: [{C_theory_2d.min():.4e}, {C_theory_2d.max():.4e}]')
    
        # Surviving deltas excluded from smooth heatmap (Θ(0) = 0 convention).
        if _td_result.get('delta_contributions'):
            print(f'  Note: {len(_td_result["delta_contributions"])} surviving '
                  f'delta(s) present but excluded from smooth heatmap.')
    
        # ── Sim heatmap: row-by-row using compute_kpoint_slice ──
        # For each τ₁ row, call compute_kpoint_slice with:
        #   lag_bins = [0, t1_bins, None]   (field a at 0, b at τ₁, c sweeps)
        from models.cumulant_estimator import compute_kpoint_slice
        heatmap_lag_bins = int(HEATMAP_TAU_MAX / dt_bin_eff)
    
        C_sim_2d = np.zeros((n_tau, n_tau))
        print(f'Evaluating sim heatmap ({n_tau} rows)...')
        for i, t1_val in enumerate(tau_1d):
            t1_bins = int(round(t1_val / dt_bin_eff))
            lag_bins_row = [0, t1_bins, None]
            try:
                tau_row, c_row = compute_kpoint_slice(
                    binned_counts, float(dt_bin_eff),
                    [int(p) for p in pop_indices],
                    lag_bins_row, heatmap_lag_bins,
                )
                C_sim_2d[i, :] = np.interp(tau_1d, tau_row, c_row)
            except Exception as e:
                print(f'  row {i} (tau1={t1_val}): {e}')
                C_sim_2d[i, :] = 0.0
        print(f'  Done. Range: [{np.nanmin(C_sim_2d):.4e}, {np.nanmax(C_sim_2d):.4e}]')
    
        # Mask sim heatmap along delta-firing lines (free-ext indices 0, 1)
        if '_td_result' in dir() and _td_result.get('delta_contributions'):
            _delta_lines = _forbidden_tau_lines_2d(
                _td_result['delta_contributions'], 1, 2, {0: 0.0})
            for (a1, a2, cf) in _delta_lines:
                for i, t1 in enumerate(tau_1d):
                    for j, t2 in enumerate(tau_1d):
                        if abs(a1 * t1 + a2 * t2 + cf) < HEATMAP_TAU_STEP * 1.0:
                            C_sim_2d[i, j] = np.nan
            print(f'  Masked {len(_delta_lines)} delta line(s) in sim heatmap.')
    
        # ── 3-panel plot ──
        fig, axes = plt.subplots(1, 3, figsize=(22, 6))
        extent = [tau_1d[0], tau_1d[-1], tau_1d[0], tau_1d[-1]]
    
        # Panel 1: Theory on its own scale
        th_max = max(abs(np.nanmin(C_theory_2d)), abs(np.nanmax(C_theory_2d)), 1e-15)
        im0 = axes[0].imshow(
            C_theory_2d.T, origin='lower', aspect='equal', extent=extent,
            cmap='RdBu_r', vmin=-th_max, vmax=th_max,
        )
        axes[0].set_xlabel(r'$\tau_1$', fontsize=13)
        axes[0].set_ylabel(r'$\tau_2$', fontsize=13)
        axes[0].set_title('Theory (tree-level)', fontsize=13)
        fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    
        # Panel 2: Sim on its own scale (shows the shot-noise ridge)
        sim_max = max(abs(np.nanmin(C_sim_2d)), abs(np.nanmax(C_sim_2d)), 1e-15)
        im1 = axes[1].imshow(
            C_sim_2d.T, origin='lower', aspect='equal', extent=extent,
            cmap='RdBu_r', vmin=-sim_max, vmax=sim_max,
        )
        axes[1].set_xlabel(r'$\tau_1$', fontsize=13)
        axes[1].set_title('Sim (full scale)', fontsize=13)
        fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    
        # Panel 3: Sim clamped to THEORY scale (reveals smooth structure)
        im2 = axes[2].imshow(
            C_sim_2d.T, origin='lower', aspect='equal', extent=extent,
            cmap='RdBu_r', vmin=-th_max, vmax=th_max,
        )
        axes[2].set_xlabel(r'$\tau_1$', fontsize=13)
        axes[2].set_title('Sim (theory scale)', fontsize=13)
        fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
        fig.suptitle(rf'$k=3$ cumulant heatmap:  $\langle {corr_label} \rangle$',
                     fontsize=14)
        plt.tight_layout()
        plt.show()
    
    else:
        print(f'Heatmap comparison only for k=3 (current k={k}).')
    